# Reproduce the Next Energy article
*How much can a battery management system know about its state of health error without capacity labels? A cross-domain evaluation on 387 lithium-ion cells*

This notebook regenerates every table (Tables 4-6 and S1-S14) and every figure (Figures 1-8, S1 and the graphical abstract) of the article, and then checks each table cell and each figure pixel against the article.

**How to run**
1. Choose *Runtime > Run all*.
2. Cell 2 copies the NASA and GMP data, the fonts used in the figures and the reference results from the GitHub repository (https://github.com/afsanaswe-pixel/Code_and_Others_for_Next_Energy). If the repository cannot be reached, it asks you to upload **`NE_data_and_reference.zip`** instead. (`00_nasa_features.py` rebuilds the NASA file from the original NASA `.mat` files; it is not needed for the run.)

No restart is needed, because every step runs in a fresh Python process with the versions installed by cell 1.

**Modes** (set `MODE` in cell 1)
- `'reference'` (default, about 10 minutes) restores the per-cycle predictions of the reported run. It then recomputes every analysis that does not train an SOH estimator and redraws all tables and figures, so the result is identical to the article.
- `'full'` (about 2 hours on the free CPU runtime) also retrains every estimator from the public data.
  - Gradient boosting, ridge regression and the statistics are identical on any computer.
  - The MLP, 1D-CNN and PINN are trained by long chains of floating-point operations, so on a different processor type their numbers can differ from the article. The verification at the end lists every such cell. Reference mode is the exact reproduction.

In [ ]:
MODE = 'reference'   # 'reference' (about 10 min, identical to the article) or 'full' (about 2 hours, retrains every model)
# Exact library versions of the reported run (Python 3.12 or newer).
!pip -q install numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 joblib==1.6.0 shap==0.54.1 statsmodels==0.15.0 jax==0.12.1 jaxlib==0.12.1 optax==0.2.9 pillow==12.3.0

In [ ]:
# Data, fonts and reference results: copied from the GitHub repository, or from an uploaded NE_data_and_reference.zip
import os, glob, shutil, subprocess, zipfile
REPO = 'https://github.com/afsanaswe-pixel/Code_and_Others_for_Next_Energy.git'
if not glob.glob('reference/outputs*.zip'):
    if not os.path.isdir('repo'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, 'repo'])
    if glob.glob('repo/reference/outputs*.zip'):
        for d in ('data', 'fonts', 'reference'):
            shutil.copytree(os.path.join('repo', d), d, dirs_exist_ok=True)
        print('copied data, fonts and reference results from', REPO)
    else:
        from google.colab import files
        print('The repository could not be reached: upload NE_data_and_reference.zip')
        up = files.upload()
        zipfile.ZipFile(next(iter(up))).extractall('.')
print('data:', sorted(os.listdir('data')))
print('fonts:', sorted(os.listdir('fonts')))
print('reference:', sorted(os.listdir('reference'))[:3], '...')

## Code
Each cell writes one script. `run_all.py` runs them in order.

In [ ]:
%%writefile common.py
"""Shared data loading and the monotone-projection certificate.

Data: processed per-cycle charge features of 387 public cells (XJTU, TJU, HUST, MIT)
as redistributed in github.com/wang-fujin/PINN4SOH (Wang et al., 2024, Nat. Commun.).
"""
import os, glob, subprocess
import numpy as np, pandas as pd
from sklearn.isotonic import IsotonicRegression

ROOT = os.path.dirname(os.path.abspath(__file__))
DATA = os.path.join(ROOT, 'data')
OUT = os.path.join(ROOT, 'outputs'); FIG = os.path.join(ROOT, 'figures')
for d in (DATA, OUT, FIG): os.makedirs(d, exist_ok=True)
PINN_REPO = 'https://github.com/wang-fujin/PINN4SOH.git'
PINN_COMMIT = 'cc3cc5053caf38f04e0665f7f88cb109144d035e'  # version used in the paper (2 Sep 2026)
QUICK = os.environ.get('MONO_QUICK', '0') == '1'
SEED = 0


def fetch_public():
    """Clone the public processed datasets once (about 265 MB)."""
    tgt = os.path.join(DATA, 'PINN4SOH')
    if not os.path.isdir(os.path.join(tgt, 'data')):
        subprocess.run(['git', 'clone', '-q', PINN_REPO, tgt], check=True)
        subprocess.run(['git', '-C', tgt, 'checkout', '-q', PINN_COMMIT], check=True)
    head = subprocess.run(['git', '-C', tgt, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
    if head and head != PINN_COMMIT:
        print(f'WARNING: data repository at {head[:10]}, paper used {PINN_COMMIT[:10]}')
    return os.path.join(tgt, 'data')


def load_cells():
    """Return one long DataFrame: one row per cycle, with columns
    16 raw features, 16 self-referenced features (d_*), soh, k (cycle index), cell, dom, dataset."""
    B = fetch_public() + '/'
    cells = []
    def add(ds, dom, f):
        d = pd.read_csv(f).replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
        feat = [c for c in d.columns if c != 'capacity']
        d['soh'] = d.capacity / d.capacity.iloc[:5].median()
        base = d[feat].iloc[:10].mean()
        for c in feat:
            d['d_' + c] = d[c] - base[c]
        d['k'] = np.arange(len(d)); d['cell'] = f'{dom}/{os.path.basename(f)}'
        d['dom'] = dom; d['dataset'] = ds
        cells.append(d)
    for f in sorted(glob.glob(B + 'XJTU data/*.csv')): add('XJTU', 'XJTU-' + os.path.basename(f).split('_battery')[0], f)
    for f in sorted(glob.glob(B + 'HUST data/*.csv')): add('HUST', 'HUST', f)
    for f in sorted(glob.glob(B + 'MIT data/*/*.csv')): add('MIT', 'MIT-' + f.split('/')[-2], f)
    for f in sorted(glob.glob(B + 'TJU data/*/*.csv')): add('TJU', 'TJU-' + f.split('/')[-2][:9], f)
    D = pd.concat(cells, ignore_index=True)
    F = [c for c in D.columns if c not in ('capacity', 'soh', 'k', 'cell', 'dom', 'dataset') and not c.startswith('d_')]
    return D, F


_iso = IsotonicRegression(increasing=False)


def project(p):
    """Euclidean projection of a trajectory onto non-increasing sequences (pool-adjacent-violators)."""
    return _iso.fit_transform(np.arange(len(p)), p)


def rms(a):
    return float(np.sqrt(np.mean(np.square(a))))


def certificate(p, y=None):
    """Return dict with projected trajectory q and label-free lower bound LB = RMS(p - q).
    Theorem (Pythagorean inequality for projections onto a closed convex set C):
    if y in C then ||p - y||^2 >= ||p - q||^2 + ||q - y||^2,
    hence LB <= RMSE(p, y) and RMSE(q, y) <= sqrt(RMSE(p, y)^2 - LB^2)."""
    q = project(p); out = dict(q=q, LB=rms(p - q))
    if y is not None:
        out.update(rmse=rms(p - y), rmse_q=rms(q - y))
    return out


def cell_rows(te, tgt, extra=None):
    """Per-cell error and label-free summaries for a held-out domain.
    te must have columns cell, dataset, soh, p (cycle-ordered within cell)."""
    rows = []
    for c, g in te.groupby('cell', sort=False):
        p, y = g.p.values, g.soh.values
        cert = certificate(p, y)
        sm = pd.Series(p).rolling(15, min_periods=1, center=True).median().values
        inc = np.diff(sm)
        r = dict(cell=c, dom=tgt, dataset=g.dataset.iloc[0], n=len(g), rmse=cert['rmse'], rmse_q=cert['rmse_q'], LB=cert['LB'],
                 bias=float(np.mean(p - y)), rebound=float(inc[inc > 0].sum() / (np.abs(inc).sum() + 1e-12)),
                 rough=float(np.std(np.diff(p)) / (np.std(p) + 1e-12)), viol=rms(y - certificate(y)['q']))
        if extra is not None: r.update(extra.get(c, {}))
        rows.append(r)
    return rows


def raw_row_counts():
    """Rows in the public CSV files before and after removal of missing/infinite values (data provenance)."""
    B = fetch_public() + '/'
    fs = glob.glob(B + 'XJTU data/*.csv') + glob.glob(B + 'HUST data/*.csv') + glob.glob(B + 'MIT data/*/*.csv') + glob.glob(B + 'TJU data/*/*.csv')
    out = []
    for f in sorted(fs):
        d = pd.read_csv(f); n0 = len(d); n1 = len(d.replace([np.inf, -np.inf], np.nan).dropna())
        out.append(dict(file=f.split('data/', 1)[1], rows_raw=n0, rows_kept=n1, removed=n0 - n1))
    return pd.DataFrame(out)


# ---------------------------------------------------------------------------------------------------------------
# Figure style. The article's figures were drawn in an environment whose matplotlib defaults differed from the
# standard ones in three settings: default font Inter, no font hinting and 200 dpi figures. They are set here
# explicitly, with the fonts shipped in fonts/ (SIL Open Font License), so that every machine (including Colab)
# draws pixel-identical figures. Importing common applies the style.
def apply_figure_style():
    try:
        import matplotlib
        from matplotlib import font_manager
    except ImportError:
        return
    for f in sorted(glob.glob(os.path.join(ROOT, 'fonts', '*.[ot]tf'))):
        font_manager.fontManager.addfont(f)
    matplotlib.rcParams.update({'font.family': ['Inter', 'sans-serif', 'DejaVu Sans'], 'text.hinting': 'no_hinting',
                                'figure.dpi': 200})


apply_figure_style()

In [ ]:
%%writefile 00_nasa_features.py
"""Step 00 (optional). Build data/nasa_cycle_features.csv from the NASA PCoE Li-ion battery aging files
B0005.mat, B0006.mat, B0007.mat and B0018.mat (Saha and Goebel, NASA Ames Prognostics Center of Excellence, Battery Data
Set, folder 1. BatteryAgingARC-FY08Q4). Put the four .mat files in data/nasa_raw/ and run: python 00_nasa_features.py
One row per discharge cycle (168, 168, 168 and 132 cycles):
  capacity  discharge capacity (Ah), field Capacity
  t_dis     time until the measured voltage first falls below 2.7 V (s)          [proxy of capacity; not used by 06]
  t_Tmax    time of the maximum measured temperature (s)                         [proxy of capacity; not used by 06]
  V_mean    mean measured voltage from the start of the record until 2.7 V (V)
  V_600s    measured voltage at 600 s, linear interpolation (V)
  T_max     maximum measured temperature (deg C)
  T_start   measured temperature at the first sample (deg C)
  I_mean    mean measured current from the start of the record until 2.7 V (A)
  T_amb     ambient temperature of the cycle (deg C)"""
import os
import numpy as np, pandas as pd
from scipy.io import loadmat

RAW = os.path.join('data', 'nasa_raw'); rows = []
for b in ['B0005', 'B0006', 'B0007', 'B0018']:
    cycles = loadmat(os.path.join(RAW, f'{b}.mat'), simplify_cells=True)[b]['cycle']; k = 0
    for c in cycles:
        if c['type'] != 'discharge':
            continue
        k += 1; d = c['data']
        t = np.asarray(d['Time'], float); v = np.asarray(d['Voltage_measured'], float)
        T = np.asarray(d['Temperature_measured'], float); I = np.asarray(d['Current_measured'], float)
        below = np.flatnonzero(v < 2.7); end = below[0] if len(below) else len(v) - 1   # first sample below 2.7 V
        rows.append(dict(battery=int(b[1:]), cycle=k, capacity=float(np.atleast_1d(d['Capacity'])[0]), t_dis=float(t[end]),
                         t_Tmax=float(t[int(np.argmax(T))]), V_mean=float(v[:end + 1].mean()), V_600s=float(np.interp(600.0, t, v)),
                         T_max=float(T.max()), T_start=float(T[0]), I_mean=float(I[:end + 1].mean()),
                         T_amb=float(c['ambient_temperature'])))
N = pd.DataFrame(rows)
N.to_csv(os.path.join('data', 'nasa_cycle_features.csv'), index=False, float_format='%.10g')
print('data/nasa_cycle_features.csv:', len(N), 'discharge cycles;', N.groupby('battery').size().to_dict())

In [ ]:
%%writefile 01_lodo.py
"""Step 1. Leave-one-domain-out (LODO) SOH estimation on 13 public domains.
Configurations: 3 learners (GBR, ridge, MLP) x 2 feature references (raw, self-referenced).
Main configuration (GBR, self-referenced) uses a 5-member bootstrap ensemble to give a
disagreement signal and a kNN feature-space distance. Writes:
  outputs/pred_<cfg>.csv.gz  per-cycle predictions on the held-out domain
  outputs/cells_<cfg>.csv    per-cell errors and label-free signals
"""
import warnings, numpy as np, pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import NearestNeighbors
from common import load_cells, certificate, rms, OUT, QUICK, SEED
warnings.filterwarnings('ignore')
D, F = load_cells()
STRIDE = 6  # training uses every 6th cycle (memory/time); evaluation uses every cycle
CFGS = [('gbr', 'self'), ('gbr', 'raw'), ('ridge', 'self'), ('ridge', 'raw'), ('mlp', 'self'), ('mlp', 'raw')]
if QUICK: CFGS = CFGS[:2]


def learner(name, s):
    if name == 'gbr': return HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, random_state=s)
    if name == 'ridge': return Ridge(alpha=1.0)
    return MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=300, early_stopping=True, n_iter_no_change=10, random_state=s)


def fold(tgt, model, ref):
    X = F if ref == 'raw' else ['d_' + c for c in F]
    tr = D[D.dom != tgt].iloc[::STRIDE]; te = D[D.dom == tgt]
    mu, sd = tr[X].mean(), tr[X].std() + 1e-12
    Ztr, Zte = ((tr[X] - mu) / sd).values, ((te[X] - mu) / sd).values
    main = (model == 'gbr' and ref == 'self')
    n_ens = 5 if main else 1
    P = []
    for s in range(n_ens):
        idx = np.random.default_rng(SEED + s).choice(len(tr), int(0.7 * len(tr)), replace=False) if n_ens > 1 else np.arange(len(tr))
        P.append(learner(model, SEED + s).fit(Ztr[idx], tr.soh.values[idx]).predict(Zte))
    P = np.array(P); te = te[['cell', 'dom', 'dataset', 'k', 'soh']].assign(p=P.mean(0), ens=P.std(0))
    if main:
        nn = NearestNeighbors(n_neighbors=5).fit(Ztr[::10])
        sub = np.arange(0, len(te), 5)
        kd = pd.Series(nn.kneighbors(Zte[sub])[0].mean(1)).groupby(te.cell.values[sub]).mean()
    rows = []
    for c, g in te.groupby('cell', sort=False):
        p, y = g.p.values, g.soh.values
        cert = certificate(p, y)
        sm = pd.Series(p).rolling(15, min_periods=1, center=True).median().values
        inc = np.diff(sm)
        r = dict(cell=c, dom=tgt, dataset=g.dataset.iloc[0], n=len(g), rmse=cert['rmse'], rmse_q=cert['rmse_q'], LB=cert['LB'],
                 bias=float(np.mean(p - y)), rebound=float(inc[inc > 0].sum() / (np.abs(inc).sum() + 1e-12)),
                 rough=float(np.std(np.diff(p)) / (np.std(p) + 1e-12)),
                 viol=rms(y - certificate(y)['q']))
        if main: r.update(ens=float(g.ens.mean()), knn=float(kd[c]))
        rows.append(r)
    return te[['cell', 'k', 'soh', 'p']], rows


for model, ref in CFGS:
    res = Parallel(n_jobs=2)(delayed(fold)(t, model, ref) for t in sorted(D.dom.unique()))
    pd.concat([r[0] for r in res]).to_csv(f'{OUT}/pred_{model}_{ref}.csv.gz', index=False, float_format='%.6f')
    C = pd.DataFrame([x for r in res for x in r[1]]); C.to_csv(f'{OUT}/cells_{model}_{ref}.csv', index=False)
    print(f'{model:5s} {ref:4s}: median cell RMSE {C.rmse.median():.4f}  mean {C.rmse.mean():.4f}  '
          f'after projection {C.rmse_q.median():.4f}  LB<=RMSE {np.mean(C.LB <= C.rmse + 1e-12):.3f}', flush=True)

In [ ]:
%%writefile 02_certificate.py
"""Step 2. Certificate and projection: offline (whole life) and online (prefix) evaluation.
Checks the theorem on every cell/configuration, the robust form LB <= RMSE + delta
(delta = RMS distance of the measured SOH from its own monotone projection),
and the Pythagorean gain RMSE_q^2 <= RMSE^2 - LB^2 (+ terms in delta)."""
import glob, warnings, numpy as np, pandas as pd
from scipy.stats import wilcoxon
from common import project, rms, OUT
import os
warnings.filterwarnings('ignore')
# the six main configurations and, when step 14 has been run, the four stronger estimators
ALL = ['gbr_self', 'gbr_raw', 'ridge_self', 'ridge_raw', 'mlp_self', 'mlp_raw', 'gbrtuned_self', 'gbrfull_self', 'cnn_self', 'pinn', 'pinnpub']
CFGS = [c for c in ALL if os.path.exists(f'{OUT}/cells_{c}.csv')]
rows = []
for cfg in CFGS:
    C = pd.read_csv(f'{OUT}/cells_{cfg}.csv')
    holds = C.LB <= C.rmse + 1e-12; robust = C.LB <= C.rmse + C.viol + 1e-12
    pyth = C.rmse_q ** 2 <= C.rmse ** 2 - C.LB ** 2 + 1e-10
    rows.append(dict(cfg=cfg, cells=len(C), med_rmse=C.rmse.median(), med_rmse_q=C.rmse_q.median(),
                     mean_rmse=C.rmse.mean(), mean_rmse_q=C.rmse_q.mean(),
                     rel_red_median=float(np.median(1 - C.rmse_q / C.rmse)),
                     improved=float((C.rmse_q <= C.rmse + 1e-12).mean()),
                     lb_holds=float(holds.mean()), lb_robust=float(robust.mean()), pyth_holds=float(pyth.mean()),
                     wilcoxon_p=float(wilcoxon(C.rmse_q, C.rmse).pvalue)))
T = pd.DataFrame(rows); T.to_csv(f'{OUT}/tab_certificate_offline.csv', index=False)
print(T.round(4).to_string(index=False))

# Online use: at checkpoint t only cycles 0..t are available. Online estimate = last value of
# the projection of the prefix (isotonic regression of the prefix).
on = []
for cfg in CFGS:
    P = pd.read_csv(f'{OUT}/pred_{cfg}.csv.gz')
    for c, g in P.groupby('cell', sort=False):
        p, y = g.p.values, g.soh.values; n = len(p)
        cps = np.unique(np.linspace(10, n - 1, 20).astype(int))
        e_raw, e_q, lbs, pre = [], [], [], []
        for t in cps:
            q = project(p[:t + 1]); e_raw.append(p[t] - y[t]); e_q.append(q[-1] - y[t])
            lbs.append(rms(p[:t + 1] - q)); pre.append(rms(p[:t + 1] - y[:t + 1]))
        on.append(dict(cfg=cfg, cell=c, rmse_online=rms(e_raw), rmse_online_q=rms(e_q),
                       lb_prefix_holds=float(np.mean(np.array(lbs) <= np.array(pre) + 1e-12))))
O = pd.DataFrame(on); O.to_csv(f'{OUT}/cells_online.csv', index=False)
S = O.groupby('cfg').agg(med_raw=('rmse_online', 'median'), med_q=('rmse_online_q', 'median'),
                         improved=('rmse_online_q', lambda s: np.nan), lb_holds=('lb_prefix_holds', 'mean'))
S['improved'] = O.assign(i=O.rmse_online_q <= O.rmse_online).groupby('cfg').i.mean()
S['wilcoxon_p'] = O.groupby('cfg').apply(lambda g: wilcoxon(g.rmse_online_q, g.rmse_online).pvalue)
S.to_csv(f'{OUT}/tab_certificate_online.csv'); print(S.round(4).to_string())

In [ ]:
%%writefile 03_signals.py
"""Step 3. Do label-free signals rank per-cell error on an unseen domain?
Main configuration (GBR, self-referenced). Signals: ensemble disagreement, kNN distance,
rebound fraction, roughness, certificate LB. Metrics: Spearman (pooled and within-domain
median) and AUROC for 'bad' cells (RMSE > 2% and > 5%), with 95% CIs from a
domain-cluster bootstrap (B = 2000)."""
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from common import OUT, SEED
C = pd.read_csv(f'{OUT}/cells_gbr_self.csv')
SIG = {'ens': 'Ensemble disagreement', 'knn': 'kNN feature distance', 'rebound': 'Rebound fraction',
       'rough': 'Roughness', 'LB': 'Certificate LB'}
doms = C.dom.unique(); rng = np.random.default_rng(SEED); B = 2000


def stats(df, s):
    within = np.nanmedian([spearmanr(g[s], g.rmse).correlation for _, g in df.groupby('dom') if len(g) > 5])
    out = dict(rho=spearmanr(df[s], df.rmse).correlation, rho_within=within)
    for thr in (0.02, 0.05):
        y = df.rmse > thr
        out[f'auc{int(thr*100)}'] = roc_auc_score(y, df[s]) if 0 < y.mean() < 1 else np.nan
    return out


rows = []
for s, name in SIG.items():
    est = stats(C, s); boot = []
    for _ in range(B):
        pick = rng.choice(doms, len(doms), replace=True)
        boot.append(stats(pd.concat([C[C.dom == d] for d in pick]), s))
    bt = pd.DataFrame(boot)
    r = dict(signal=name)
    for k, v in est.items():
        lo, hi = np.nanpercentile(bt[k], [2.5, 97.5]); r[k] = v; r[k + '_lo'] = lo; r[k + '_hi'] = hi
    rows.append(r)
T = pd.DataFrame(rows); T.to_csv(f'{OUT}/tab_signals.csv', index=False)
print(T.round(3).to_string(index=False))
print('bad-cell prevalence: >2pct %.3f, >5pct %.3f' % ((C.rmse > .02).mean(), (C.rmse > .05).mean()))
# error decomposition: share of squared error explained by the per-cell mean bias
C['bias_share'] = C.bias ** 2 / C.rmse ** 2
print('median share of MSE due to constant per-cell bias: %.3f' % C.bias_share.median())
pd.DataFrame({'bias_share_median': [C.bias_share.median()], 'bias_share_mean': [C.bias_share.mean()]}).to_csv(f'{OUT}/tab_bias_share.csv', index=False)

In [ ]:
%%writefile 04_meta.py
"""Step 4. Can a learned meta-model turn label-free signals into a calibrated per-cell
error estimate on a new domain? Nested leave-one-domain-out: for each target domain the
meta-model (GBR on log signals -> log RMSE) and a split-conformal upper bound (90%) are fit
only on the remaining 12 domains (conformity scores from an inner LODO)."""
import numpy as np, pandas as pd, warnings
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from common import OUT
warnings.filterwarnings('ignore')
C = pd.read_csv(f'{OUT}/cells_gbr_self.csv')
C = C.assign(lLB=np.log(C.LB + 1e-6), lens=np.log(C.ens), lknn=np.log(C.knn), y=np.log(C.rmse))
FE = ['lLB', 'lens', 'lknn', 'rebound', 'rough']
ALPHA = 0.10
mk = {'gbr': lambda: HistGradientBoostingRegressor(max_iter=150, max_depth=3, learning_rate=0.05, min_samples_leaf=10),
      'ridge': lambda: Ridge(1.0)}
res = []
for name, m in mk.items():
    out = []
    for t in C.dom.unique():
        tr, te = C[C.dom != t], C[C.dom == t]
        scores = []
        for u in tr.dom.unique():
            a, b = tr[tr.dom != u], tr[tr.dom == u]
            scores += list(b.y - m().fit(a[FE], a.y).predict(b[FE]))
        q = np.quantile(scores, np.ceil((1 - ALPHA) * (len(scores) + 1)) / len(scores))
        pr = m().fit(tr[FE], tr.y).predict(te[FE])
        out.append(te.assign(pred=np.exp(pr), UB=np.exp(pr + q)))
    O = pd.concat(out)
    cov_dom = O.groupby('dom').apply(lambda g: (g.rmse <= g.UB).mean())
    res.append(dict(meta=name, rho=spearmanr(O.pred, O.rmse).correlation,
                    rho_within=np.nanmedian([spearmanr(g.pred, g.rmse).correlation for _, g in O.groupby('dom')]),
                    coverage=(O.rmse <= O.UB).mean(), cov_min_domain=cov_dom.min(), cov_min_dom_name=cov_dom.idxmin(),
                    n_domains_under_80=int((cov_dom < 0.8).sum()), ub_over_rmse_median=float((O.UB / O.rmse).median()),
                    mae_est=float(np.median(np.abs(O.pred - O.rmse))),
                    mae_constant=float(np.median(np.abs(C.rmse.median() - C.rmse)))))
    O.to_csv(f'{OUT}/cells_meta_{name}.csv', index=False)
    print(name, cov_dom.round(2).to_dict())
T = pd.DataFrame(res); T.to_csv(f'{OUT}/tab_meta.csv', index=False); print(T.round(3).to_string(index=False))

In [ ]:
%%writefile 05_checks.py
"""Step 5. Sparse reference capacity checks (RPTs) combined with the projection.
For K checks per cell the residual (y - p) is measured at K cycles, linearly interpolated
(constant for K = 1) and added to p before projection; error is evaluated on non-check cycles.
Placements compared at equal K: periodic (equal cycle spacing), equal predicted-fade spacing
(label-free), largest local departure from monotonicity (label-free)."""
import numpy as np, pandas as pd
from scipy.stats import wilcoxon
from common import project, rms, OUT
P = pd.read_csv(f'{OUT}/pred_gbr_self.csv.gz')


def corrected(p, idx, y):
    if len(idx) == 0: return project(p)
    r = y[idx] - p[idx]
    c = np.full(len(p), r[0]) if len(idx) == 1 else np.interp(np.arange(len(p)), idx, r)
    return project(p + c)


rows = []
for c, g in P.groupby('cell', sort=False):
    p, y = g.p.values, g.soh.values; n = len(p); q0 = project(p)
    rec = dict(cell=c, dom=c.split('/')[0], K0=rms(q0 - y), base=rms(p - y))
    for K in range(1, 6):
        place = {'periodic': np.unique(np.linspace(0, n - 1, K + 2)[1:-1].astype(int))}
        fade = 1 - q0; tf = np.linspace(fade.min(), fade.max(), K + 2)[1:-1]
        place['fade'] = np.unique([int(np.argmin(np.abs(fade - f))) for f in tf])
        dev = pd.Series(np.abs(p - q0)).rolling(max(5, n // 20), center=True, min_periods=1).mean().values.copy()
        sel = []
        for _ in range(K):
            j = int(np.argmax(dev)); sel.append(j); dev[max(0, j - n // (K + 1)):j + n // (K + 1)] = -1
        place['departure'] = np.unique(sel)
        for nm, idx in place.items():
            idx = np.sort(idx); m = np.ones(n, bool); m[idx] = False
            rec[f'{nm}{K}'] = rms(corrected(p, idx, y)[m] - y[m])
    rows.append(rec)
R = pd.DataFrame(rows); R.to_csv(f'{OUT}/cells_checks.csv', index=False)
summ = []
for K in range(0, 6):
    if K == 0:
        summ.append(dict(K=0, placement='none', median=R.K0.median(), mean=R.K0.mean())); continue
    for nm in ['periodic', 'fade', 'departure']:
        col = f'{nm}{K}'
        d = dict(K=K, placement=nm, median=R[col].median(), mean=R[col].mean())
        if nm != 'periodic':
            d['better_than_periodic'] = float((R[col] < R[f'periodic{K}']).mean())
            d['wilcoxon_p_vs_periodic'] = float(wilcoxon(R[col], R[f'periodic{K}']).pvalue)
        summ.append(d)
S = pd.DataFrame(summ); S.to_csv(f'{OUT}/tab_checks.csv', index=False); print(S.round(4).to_string(index=False))
print('base median %.4f mean %.4f' % (R.base.median(), R.base.mean()))

In [ ]:
%%writefile 06_controls.py
"""Step 6. Known-truth and assumption-violation controls.
(a) GMP pack (Mendeley doi:10.17632/k675pxrd83.2): exact monotone ground truth.
    (a1) a learner trained on the operating channels, chronological 70/30 split;
    (a2) controlled error injection: p = y + b + e, e ~ AR(1)(phi) with sd s, on the
         true 3000-cycle trajectory, to show which error component the certificate sees.
(b) NASA B0005/6/7/18: measured capacity with regeneration (truth not monotone);
    leave-one-cell-out with non-proxy discharge features."""
import os, numpy as np, pandas as pd, warnings
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from common import certificate, project, rms, DATA, OUT, SEED
warnings.filterwarnings('ignore')
rng = np.random.default_rng(SEED)

# ---------- (a) GMP ----------
gp = os.path.join(DATA, 'GMP.csv')
if not os.path.exists(gp):  # regenerate from the published generator (Ulfat et al., Data in Brief 69 (2026) 113252; seed 42)
    np.random.seed(42); N = 3000; n = np.arange(1, N + 1); cap = 40.0 - 12.0 * (n / N) ** 1.35
    V = np.random.normal(25.60, 0.12, N); I_ = np.random.uniform(-40.0, 40.0, N)
    Tt = np.random.normal(28.0, 1.8, N); Cv = np.random.normal(3.200, 0.008, (N, 8))
    g = pd.DataFrame({'PackID': 1, 'CycleNumber': n, 'Capacity(Ah)': cap, 'SOH(%)': cap / 40 * 100, 'RUL(cycles)': N - n,
                      'Voltage(V)': V, 'Current(A)': I_, 'DoD(%)': n / N * 100, 'Temperature(C)': Tt})
    for i in range(8): g[f'Cell{i+1}_V'] = Cv[:, i]
    g.to_csv(gp, index=False)
G = pd.read_csv(gp); y_g = (G['Capacity(Ah)'] / G['Capacity(Ah)'].iloc[:5].median()).values
ops = ['Voltage(V)', 'Current(A)', 'Temperature(C)'] + [f'Cell{i}_V' for i in range(1, 9)]
cut = int(0.7 * len(G))
m = HistGradientBoostingRegressor(max_iter=200, random_state=SEED).fit(G[ops].iloc[:cut], y_g[:cut])
c = certificate(m.predict(G[ops].iloc[cut:]), y_g[cut:])
a1 = dict(case='GMP learner on operating channels (chronological 70/30)', rmse=c['rmse'], rmse_q=c['rmse_q'], LB=c['LB'],
          viol=rms(y_g - project(y_g)))
inj = []
for b in [0.0, 0.02, 0.05]:
    for s in [0.005, 0.01, 0.02]:
        for phi in [0.0, 0.9]:
            for rep in range(20):
                e = np.zeros(len(y_g)); z = rng.normal(0, s * np.sqrt(1 - phi ** 2), len(y_g)); e[0] = rng.normal(0, s)
                for t in range(1, len(e)): e[t] = phi * e[t - 1] + z[t]
                cc = certificate(y_g + b + e, y_g)
                inj.append(dict(bias=b, noise_sd=s, phi=phi, rep=rep, rmse=cc['rmse'], rmse_q=cc['rmse_q'], LB=cc['LB']))
I = pd.DataFrame(inj); I.to_csv(f'{OUT}/gmp_injection.csv', index=False)
Ig = I.groupby(['bias', 'noise_sd', 'phi'])[['rmse', 'rmse_q', 'LB']].mean().reset_index()
Ig.to_csv(f'{OUT}/tab_gmp_injection.csv', index=False)
print(pd.DataFrame([a1]).round(4).to_string(index=False)); print(Ig.round(4).to_string(index=False))

# ---------- (b) NASA ----------
npth = os.path.join(DATA, 'nasa_cycle_features.csv')
N = pd.read_csv(npth)
X = ['V_mean', 'V_600s', 'T_max', 'T_start', 'I_mean']  # discharge-time proxies excluded
for b_, g in N.groupby('battery'):
    base = g[X].iloc[:5].mean(); N.loc[g.index, [f'd_{x}' for x in X]] = (g[X] - base).values
    N.loc[g.index, 'soh'] = g.capacity / g.capacity.iloc[:5].median()
XD = [f'd_{x}' for x in X]; rows = []
for name, mk in [('gbr', lambda: HistGradientBoostingRegressor(max_iter=200, random_state=SEED)), ('ridge', lambda: Ridge(1.0))]:
    for b_ in N.battery.unique():
        tr, te = N[N.battery != b_], N[N.battery == b_]
        p = mk().fit(tr[XD], tr.soh).predict(te[XD]); yv = te.soh.values; cc = certificate(p, yv)
        dv = rms(yv - project(yv))
        rows.append(dict(learner=name, cell=f'B{int(b_):04d}', rmse=cc['rmse'], rmse_q=cc['rmse_q'], LB=cc['LB'], viol=dv,
                         lb_holds=cc['LB'] <= cc['rmse'], lb_robust=cc['LB'] <= cc['rmse'] + dv,
                         improved=cc['rmse_q'] <= cc['rmse'], improved_robust=cc['rmse_q'] <= cc['rmse'] + 2 * dv))
NN = pd.DataFrame(rows); NN.to_csv(f'{OUT}/tab_nasa.csv', index=False); print(NN.round(4).to_string(index=False))
pd.DataFrame([a1]).to_csv(f'{OUT}/tab_gmp.csv', index=False)

# ---------- (a3) dense bias x noise surface on GMP (for the 3-D panel of Figure 4) ----------
surf = []
for b in np.linspace(0, 0.05, 11):
    for s in np.linspace(0.0025, 0.03, 12):
        for rep in range(5):
            cc = certificate(y_g + b + rng.normal(0, s, len(y_g)), y_g)
            surf.append(dict(bias=b, noise_sd=s, rep=rep, rmse=cc['rmse'], rmse_q=cc['rmse_q'], LB=cc['LB']))
S = pd.DataFrame(surf).groupby(['bias', 'noise_sd'])[['rmse', 'rmse_q', 'LB']].mean().reset_index()
S['removed'] = S.LB ** 2 / S.rmse ** 2; S.to_csv(f'{OUT}/gmp_surface.csv', index=False)
print('surface: removed share range %.3f-%.3f' % (S.removed.min(), S.removed.max()))

In [ ]:
%%writefile 07_figures.py
"""Step 7. Data figures from outputs/, saved under the Next Energy article numbering:
Figure_2 (unseen-domain error), Figure_3 (certificate), Figure_4 (GMP known truth), Figure_5 (conformal bounds),
Figure_6 (reference checks). Two further plots are not in the article and are saved as not_in_paper_*.png:
the label-free warning signals (values in Table 5) and the offline check placements (values in Table S4)."""
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from common import OUT, FIG, project
plt.rcParams.update({'font.size': 13, 'font.weight': 'bold', 'axes.labelweight': 'bold', 'axes.titleweight': 'bold',
                     'axes.titlesize': 14, 'axes.labelsize': 13, 'xtick.labelsize': 12, 'ytick.labelsize': 12,
                     'legend.fontsize': 11, 'axes.spines.top': False, 'axes.spines.right': False, 'savefig.dpi': 300})
BLUE, ORANGE, GREEN, AMBER, PINK, GREY, INK = '#1f5fbf', '#d9541e', '#138a5e', '#c98a00', '#c2477d', '#8a8a85', '#111111'
import textwrap
def title(a, t):
    w = a.get_position().width * a.figure.get_figwidth()
    a.set_title(textwrap.fill(t, max(20, int(w * 72 / (14 * 0.6)))), loc='left', pad=8)
CFG = ['gbr_self', 'gbr_raw', 'mlp_self', 'mlp_raw', 'ridge_self', 'ridge_raw']
CFG10 = ['gbr_self', 'gbr_raw', 'ridge_self', 'ridge_raw', 'mlp_self', 'mlp_raw', 'gbrtuned_self', 'gbrfull_self', 'cnn_self', 'pinn', 'pinnpub']
LAB = {'gbr_self': 'GBR, self-ref.', 'gbr_raw': 'GBR, raw', 'mlp_self': 'MLP, self-ref.', 'mlp_raw': 'MLP, raw',
       'ridge_self': 'Ridge, self-ref.', 'ridge_raw': 'Ridge, raw', 'gbrtuned_self': 'Tuned GBR, self-ref.',
       'gbrfull_self': 'GBR, all cycles', 'cnn_self': '1D-CNN, self-ref.', 'pinn': 'PINN (Wang et al.)', 'pinnpub': 'PINN, published scaling*'}
COL = {'gbr_self': BLUE, 'gbr_raw': BLUE, 'mlp_self': ORANGE, 'mlp_raw': ORANGE, 'ridge_self': GREEN, 'ridge_raw': GREEN,
       'gbrtuned_self': '#0b2a5b', 'gbrfull_self': '#5b8fd9', 'cnn_self': PINK, 'pinn': AMBER, 'pinnpub': '#8a6d00'}

# ---------- Fig. 2: unseen-domain error by domain and estimator ----------
cells = {c: pd.read_csv(f'{OUT}/cells_{c}.csv') for c in CFG10}
M = pd.DataFrame({LAB[c]: cells[c].groupby('dom').rmse.median() * 100 for c in CFG10})
fig, a = plt.subplots(figsize=(14, 7.6))
from matplotlib.colors import LinearSegmentedColormap
cm = LinearSegmentedColormap.from_list('l', plt.get_cmap('YlOrBr')(np.linspace(0, .6, 256)))
im = a.imshow(M.values, cmap=cm, aspect='auto', vmin=0, vmax=12)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        a.text(j, i, f'{M.values[i, j]:.1f}', ha='center', va='center', color=INK, fontsize=11)
a.set_xticks(range(M.shape[1])); a.set_xticklabels(M.columns, rotation=30, ha='right'); a.axvline(5.5, color=INK, lw=3)
a.set_yticks(range(M.shape[0])); a.set_yticklabels(M.index)
cb = fig.colorbar(im, ax=a, fraction=0.04); cb.set_label('median per-cell RMSE (% SOH)')
title(a, 'Unseen-domain SOH error varies more across domains than across eleven estimator configurations, including tuned, sequence and physics-informed models')
fig.tight_layout(); fig.savefig(f'{FIG}/Figure_2.png'); plt.close(fig)

# ---------- Fig. 3: certificate geometry and validity ----------
P = pd.read_csv(f'{OUT}/pred_mlp_raw.csv.gz'); Cm = cells['mlp_raw']
cand = Cm[(Cm.viol < 0.001) & (Cm.rmse < 0.08)]; ex = cand.sort_values('LB').iloc[int(0.85 * len(cand))].cell
g = P[P.cell == ex]; p, y = g.p.values, g.soh.values; q = project(p)
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
fig = plt.figure(figsize=(15.5, 12))
# (a) 3-D geometry of the certificate for n = 3 cycles
a = fig.add_subplot(2, 2, 1, projection='3d')
u = np.linspace(0.4, 1, 25); U, T = np.meshgrid(u, u); M = T <= U
P1 = np.where(M, U, np.nan); P2 = np.where(M, T, np.nan)
a.plot_surface(P1, P1, P2, color='#a9c6f0', alpha=.30, linewidth=0)          # face q1 = q2 >= q3
a.plot_surface(P1, P2, P2, color='#9fd6b8', alpha=.30, linewidth=0)          # face q1 >= q2 = q3
a.plot([0.4, 1], [0.4, 1], [0.4, 1], color=INK, lw=2)
yv = np.array([0.95, 0.78, 0.50]); pv = np.array([0.62, 0.92, 0.58]); qv = project(pv)
a.plot(*np.c_[pv, qv], color=ORANGE, lw=5, label='p \u2192 q : certificate LB')
a.plot(*np.c_[qv, yv], color=BLUE, lw=5, label='q \u2192 y : error after projection')
a.plot(*np.c_[pv, yv], color=GREY, lw=3, ls='--', label='p \u2192 y : error before')
a.scatter(*pv, s=260, color=ORANGE, depthshade=False, edgecolor=INK, linewidth=1.5, label='prediction p (outside C)')
a.scatter(*qv, s=260, color=BLUE, depthshade=False, edgecolor=INK, linewidth=1.5, label='projection q (on C)')
a.scatter(*yv, s=260, color=INK, depthshade=False, edgecolor='white', linewidth=1.5, label='true y (inside C)')
a.legend(loc='upper left', bbox_to_anchor=(-0.02, 0.93), frameon=False, fontsize=11)
a.set_xlabel('q1', labelpad=8); a.set_ylabel('q2', labelpad=8); a.set_zlabel('q3', labelpad=8)
a.set_xlim(0.4, 1); a.set_ylim(0.4, 1); a.set_zlim(0.4, 1); a.view_init(elev=24, azim=-62)
a.set_title('(a) 3-D geometry for three cycles, C = {q1 \u2265 q2 \u2265 q3}:\n|p\u2212y|\u00b2 \u2265 |p\u2212q|\u00b2 + |q\u2212y|\u00b2, hence LB \u2264 error', loc='left', pad=4)
a = fig.add_subplot(2, 2, 2)
a.plot(g.k, p, color=ORANGE, lw=1.2, label='prediction p'); a.plot(g.k, q, color=BLUE, lw=2.6, label='projection q')
a.plot(g.k, y, color=INK, lw=2, ls='--', label='measured SOH y'); a.set_xlabel('cycle'); a.set_ylabel('SOH'); a.legend(frameon=False)
title(a, f'(b) The projection q removes rebounds of the prediction p (cell from {ex.split("/")[0]})')
a = fig.add_subplot(2, 2, 3)
for c in CFG10:
    d = cells[c]; a.scatter(d.rmse * 100, d.LB * 100, s=9, color=COL[c], alpha=.4, label=LAB[c].split(',')[0] if c in ('gbr_self', 'mlp_self', 'ridge_self', 'gbrtuned_self', 'cnn_self', 'pinn') else None)
mx = max(d.rmse.max() for d in cells.values()) * 100; a.plot([0.05, mx], [0.05, mx], color=INK, lw=2, ls='--'); a.set_xscale('log'); a.set_yscale('log'); a.set_ylim(0.01, mx * 1.5); a.set_xlim(0.1, mx * 1.5)
a.set_xlabel('true per-cell RMSE (% SOH)'); a.set_ylabel('certificate LB (% SOH)'); a.legend(frameon=False, markerscale=2, fontsize=10, ncol=2, loc='lower right')
nv = sum(int((d.LB <= d.rmse + 1e-12).sum()) for d in cells.values()); nt = sum(len(d) for d in cells.values())
title(a, f'(c) LB stayed below the true error in {nv:,} of {nt:,} cell-estimator pairs (log scale)')
a = fig.add_subplot(2, 2, 4)
for c in CFG10:
    d = cells[c]; a.scatter(d.rmse * 100, d.rmse_q * 100, s=9, color=COL[c], alpha=.4)
a.plot([0.1, mx], [0.1, mx], color=INK, lw=2, ls='--'); a.set_xscale('log'); a.set_yscale('log'); a.set_xlabel('RMSE before projection (% SOH)'); a.set_ylabel('RMSE after projection (% SOH)')
imp = [(d.rmse_q <= d.rmse + 1e-12).mean() * 100 for d in cells.values()]
title(a, f'(d) Projection lowered the error in {min(imp):.0f}-{max(imp):.0f}% of cells for every estimator (log scale)')
fig.tight_layout(); fig.savefig(f'{FIG}/Figure_3.png'); plt.close(fig)

# ---------- Fig. 4: GMP known-truth injection ----------
I = pd.read_csv(f'{OUT}/tab_gmp_injection.csv'); I0 = I[I.phi == 0]
fig = plt.figure(figsize=(16, 12.5)); ax = [fig.add_subplot(2, 2, 1), fig.add_subplot(2, 2, 2)]
a = ax[0]
for b, col, mk in [(0, BLUE, 'o'), (0.02, ORANGE, 's'), (0.05, GREEN, '^')]:
    d = I0[I0.bias == b]; a.plot(d.noise_sd * 100, d.rmse * 100, color=col, marker=mk, lw=2.5, ms=8, label=f'true RMSE, bias {b*100:.0f}%')
    a.plot(d.noise_sd * 100, d.LB * 100, color=col, marker=mk, lw=2.5, ms=8, ls=':', mfc='white')
a.set_xlabel('injected noise SD (% SOH)'); a.set_ylabel('% SOH'); a.legend(frameon=False, title='dotted = certificate LB', loc='center left', bbox_to_anchor=(0.0, 0.62))
title(a, '(a) The certificate measures the noise and is blind to bias')
a = ax[1]; w = 0.27; xs = np.arange(3)
for i, (b, col) in enumerate([(0, BLUE), (0.02, ORANGE), (0.05, GREEN)]):
    d = I0[I0.bias == b]; a.bar(xs + (i - 1) * w, (1 - d.rmse_q / d.rmse).values * 100, w, color=col, label=f'bias {b*100:.0f}%')
a.set_xticks(xs); a.set_xticklabels([f'noise {s*100:.1f}%' for s in sorted(I0.noise_sd.unique())])
a.set_ylabel('RMSE reduction by projection (%)'); a.legend(frameon=False)
title(a, '(b) Projection removes noise, so its gain shrinks as bias dominates')
from matplotlib.colors import LinearSegmentedColormap
cm3 = LinearSegmentedColormap.from_list('l3', plt.get_cmap('YlOrBr')(np.linspace(0.05, .6, 256)))
for k, (fn, lab) in enumerate([('gmp_surface.csv', '(c) Independent noise'), ('gmp_surface_ar1.csv', '(d) Autocorrelated noise, AR(1) \u03c6 = 0.9')]):
    a = fig.add_subplot(2, 2, 3 + k, projection='3d'); Sf = pd.read_csv(f'{OUT}/{fn}')
    Bv = np.sort(Sf.bias.unique()); Nv = np.sort(Sf.noise_sd.unique()); Z = Sf.pivot(index='noise_sd', columns='bias', values='removed').loc[Nv, Bv].values * 100
    BB, NN = np.meshgrid(Bv * 100, Nv * 100)
    a.plot_surface(BB, NN, Z, cmap=cm3, edgecolor='#555555', linewidth=0.3, alpha=0.95, vmin=0, vmax=100)
    a.set_xlabel('bias (% SOH)', labelpad=10); a.set_ylabel('noise SD (% SOH)', labelpad=10); a.set_zlabel('error visible (%)', labelpad=10)
    a.view_init(elev=26, azim=-130); a.set_zlim(0, 100)
    a.set_title(lab + ':\nshare of squared error visible to the certificate (LB\u00b2/RMSE\u00b2)', loc='left', pad=4)
fig.tight_layout(); fig.savefig(f'{FIG}/Figure_4.png'); plt.close(fig)

# ---------- not in the article (values in Table 5): label-free warning signals ----------
S = pd.read_csv(f'{OUT}/tab_signals.csv')
fig, ax = plt.subplots(1, 2, figsize=(15, 5.6), sharey=True)
yy = np.arange(len(S))[::-1]
for a, (k, lab, ref) in zip(ax, [('auc2', 'AUROC for RMSE > 2% SOH', 0.5), ('rho_within', 'within-domain Spearman with RMSE', 0)]):
    a.errorbar(S[k], yy, xerr=[S[k] - S[k + '_lo'], S[k + '_hi'] - S[k]], fmt='o', color=BLUE, ms=10, lw=3, capsize=6)
    a.axvline(ref, color=INK, ls='--', lw=2); a.set_xlabel(lab); a.set_yticks(yy); a.set_yticklabels(S.signal)
title(ax[0], '(a) Signals separate bad from good cells only with wide uncertainty')
title(ax[1], '(b) Within a domain, ranking of cells by error is weak (95% domain-bootstrap CIs)')
fig.tight_layout(); fig.savefig(f'{FIG}/not_in_paper_warning_signals.png'); plt.close(fig)

# ---------- Fig. 5: conformal error bounds ----------
Q = pd.read_csv(f'{OUT}/tab_conformal_domains.csv'); Cf = pd.read_csv(f'{OUT}/cells_conformal.csv')
order = Q[Q.variant == 'split'].sort_values('coverage').dom.tolist()
fig, ax = plt.subplots(1, 2, figsize=(16, 6.4), gridspec_kw=dict(width_ratios=[1.35, 1]))
a = ax[0]; yy = {d: i for i, d in enumerate(order)}
for k, (v, col, mk, lab) in enumerate([('split', ORANGE, 'o', 'split (pooled)'), ('weighted', PINK, 's', 'weighted (covariate shift)'),
                                       ('family', GREEN, 'D', 'group (dataset family)'), ('hier', BLUE, '^', 'hierarchical (domain level)')]):
    d = Q[Q.variant == v].set_index('dom').loc[order]; y = np.arange(len(order)) + (k - 1.5) * 0.18
    a.errorbar(d.coverage * 100, y, xerr=[(d.coverage - d.lo) * 100, (d.hi - d.coverage) * 100], fmt=mk, color=col, ms=7, lw=1.4, capsize=2, label=lab)
a.axvline(90, color=INK, ls='--', lw=2); a.set_yticks(range(len(order))); a.set_yticklabels(order, fontsize=11)
a.set_xlabel('coverage of the 90% upper bound (%, 95% Clopper-Pearson)'); a.set_xlim(-2, 104)
a.legend(frameon=False, fontsize=10.5, loc='upper left')
title(a, '(a) Per-domain coverage: only the weighted variant reaches HUST, the most dissimilar domain')
a = ax[1]; T6 = pd.read_csv(f'{OUT}/tab_conformal.csv').set_index('variant').loc[['split', 'weighted', 'family', 'hier']]
lbl = ['split', 'weighted', 'group', 'hierarchical']; x = np.arange(4)
a.bar(x - 0.2, T6.coverage * 100, 0.4, color=BLUE, label='overall coverage (%)')
a.bar(x + 0.2, T6.infinite_bounds * 100, 0.4, color=ORANGE, label='non-informative (infinite) bounds (%)')
for i, (c_, w_) in enumerate(zip(T6.coverage, T6.width_median)): a.text(i - 0.2, c_ * 100 + 1.5, f'{c_*100:.0f}', ha='center', fontsize=11); a.text(i, -9, f'width {w_:.1f}x', ha='center', fontsize=10.5, color=INK)
a.axhline(90, color=INK, ls='--', lw=2); a.set_xticks(x); a.set_xticklabels(lbl); a.set_ylim(-12, 105); a.legend(frameon=False, fontsize=10.5, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=1)
title(a, '(b) Coverage is bought with wider or non-informative bounds')
fig.tight_layout(); fig.savefig(f'{FIG}/Figure_5.png'); plt.close(fig)

# ---------- Fig. 6: deployable (causal) reference-check schedules ----------
T7 = pd.read_csv(f'{OUT}/tab_checks_causal.csv')
fig, a = plt.subplots(figsize=(11, 6.0))
b0 = T7[T7.K == 0]['median'].values[0] * 100
for nm, col, mk, ls, lab in [('fixed', BLUE, 'o', '-', 'fixed interval (source-domain life)'), ('event', ORANGE, '^', '-', 'event-triggered (departure, causal)'),
                             ('periodic_known', GREY, 's', '--', 'periodic with known life (reference)')]:
    d = T7[T7.schedule == nm]; a.plot([0] + list(d.K), [b0] + list(d['median'] * 100), color=col, marker=mk, lw=3, ms=10, ls=ls, label=lab)
a.set_xlabel('reference capacity checks per cell, K (budget)'); a.set_ylabel('median online per-cell RMSE (% SOH)'); a.legend(frameon=False)
title(a, 'Online, a few measured checks lower the error; knowing when to check matters as much as how many')
fig.tight_layout(); fig.savefig(f'{FIG}/Figure_6.png'); plt.close(fig)

# ---------- not in the article (values in Table S4): offline placement comparison ----------
T = pd.read_csv(f'{OUT}/tab_checks.csv')
fig, a = plt.subplots(figsize=(10, 5.8))
base = T[T.K == 0]['median'].values[0] * 100
for nm, col, mk, lab in [('periodic', BLUE, 'o', 'periodic (equal cycles)'), ('fade', AMBER, 's', 'equal predicted fade'),
                         ('departure', ORANGE, '^', 'largest departure from monotone')]:
    d = T[T.placement == nm]; a.plot([0] + list(d.K), [base] + list(d['median'] * 100), color=col, marker=mk, lw=3, ms=10, label=lab)
a.set_xlabel('reference capacity checks per cell, K'); a.set_ylabel('median per-cell RMSE (% SOH)'); a.legend(frameon=False)
title(a, 'Offline placements that use the whole predicted trajectory (not deployable)')
fig.tight_layout(); fig.savefig(f'{FIG}/not_in_paper_offline_check_placements.png'); plt.close(fig)
print('figures done')

In [ ]:
%%writefile 08_workflow.py
"""Figure 1: methodology figure. Layered pipeline whose icons are miniatures of the paper's own
figures and tables, redrawn from outputs/ (run after steps 01-12). Dark bold text on light fills only."""
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.colors import LinearSegmentedColormap
from common import OUT, FIG, project
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.weight': 'bold'})
INK, NAVY, GREY = '#111111', '#0b2a5b', '#8a8a85'
BLUE, ORANGE, GREEN, AMBER, PINK = '#1f5fbf', '#d9541e', '#138a5e', '#c98a00', '#c2477d'
TINT = {BLUE: '#dde8f8', ORANGE: '#fbe4d8', GREEN: '#dff0e8', AMBER: '#f6f1df', PINK: '#f7e9f0', GREY: '#eeeeea'}
CM = LinearSegmentedColormap.from_list('l', plt.get_cmap('YlOrBr')(np.linspace(0, .6, 256)))
W, H = 160, 92
fig = plt.figure(figsize=(16, 9.2), dpi=300)
ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, W); ax.set_ylim(0, H); ax.axis('off')


def panel(x, y, w, h, col, head):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0,rounding_size=1.4', fc='white', ec=col, lw=2.6, zorder=1))
    ax.add_patch(FancyBboxPatch((x, y + h - 5.2), w, 5.2, boxstyle='round,pad=0,rounding_size=1.4', fc=TINT[col], ec=col, lw=2.6, zorder=1))
    ax.text(x + w / 2, y + h - 2.6, head, ha='center', va='center', fontsize=13.5, color=INK, zorder=3)


def mini(x, y, w, h, tag=None):
    a = fig.add_axes([x / W, y / H, w / W, h / H]); a.set_xticks([]); a.set_yticks([])
    for s in a.spines.values(): s.set_color(GREY); s.set_linewidth(1)
    if tag: ax.text(x + w, y - 0.4, tag, ha='right', va='top', fontsize=9.5, color=NAVY)
    return a


def chip(x, y, t, fs=10.8, c=INK):
    ax.text(x, y, t, ha='center', va='center', fontsize=fs, color=c, zorder=4,
            bbox=dict(boxstyle='round,pad=0.22', fc='#f3f1ea', ec='none'))


def arrow(p, q, c=INK, lw=2.6):
    ax.add_patch(FancyArrowPatch(p, q, arrowstyle='-|>', mutation_scale=20, lw=lw, color=c, zorder=5))


TOPY, TOPH, PW, GAP, X0 = 33, 57, 29.6, 2.5, 1.5
xs = [X0 + i * (PW + GAP) for i in range(5)]
heads = ['1  Public data', '2  Estimation', '3  Label-free monitoring', '4  Calibration', '5  Trustworthiness']
cols = [GREEN, BLUE, ORANGE, PINK, AMBER]
for x, hd, c in zip(xs, heads, cols): panel(x, TOPY, PW, TOPH, c, hd)
for i in range(4): arrow((xs[i] + PW + 0.2, TOPY + 30), (xs[i + 1] - 0.2, TOPY + 30))

# ---- 1 data: battery icons + mini table ----
x = xs[0]
for k, (nm, c) in enumerate([('XJTU', BLUE), ('TJU', ORANGE), ('HUST', GREEN), ('MIT', PINK)]):
    bx = x + 2.2 + k * 6.6
    ax.add_patch(FancyBboxPatch((bx, 70), 5.0, 10.5, boxstyle='round,pad=0,rounding_size=0.8', fc=TINT[c], ec=c, lw=2))
    ax.add_patch(Rectangle((bx + 1.6, 80.5), 1.8, 1.0, fc=c, ec='none'))
    ax.add_patch(Rectangle((bx + 0.7, 71), 3.6, 6.5 - k * 1.2, fc=c, ec='none', alpha=.8))
    ax.text(bx + 2.5, 68.3, nm, ha='center', fontsize=10, color=INK)
cells = [['domain', 'cells', 'cycles'], ['13', '387', '310,554']]
for r in range(2):
    for cc in range(3):
        ax.add_patch(Rectangle((x + 2.5 + cc * 8.2, 57.5 - r * 4), 8.2, 4, fc='#e7eef7' if r == 0 else 'white', ec=GREY, lw=1))
        ax.text(x + 6.6 + cc * 8.2, 59.5 - r * 4, cells[r][cc], ha='center', va='center', fontsize=10, color=INK)
ax.text(x + 27.1, 52.6, 'Table 2', ha='right', va='top', fontsize=9.5, color=NAVY)
chip(x + PW / 2, 47.5, '16 charge features per cycle')
chip(x + PW / 2, 43.2, 'SOH = C / C(start)')
chip(x + PW / 2, 38.9, 'controls: GMP, NASA')

# ---- 2 estimation: mini heatmap (Fig. 2) ----
x = xs[1]
CFG = ['gbr_self', 'gbr_raw', 'mlp_self', 'mlp_raw', 'ridge_self', 'ridge_raw']
CFG10 = ['gbr_self', 'gbr_raw', 'ridge_self', 'ridge_raw', 'mlp_self', 'mlp_raw', 'gbrtuned_self', 'gbrfull_self', 'cnn_self', 'pinn', 'pinnpub']
M = pd.DataFrame({c: pd.read_csv(f'{OUT}/cells_{c}.csv').groupby('dom').rmse.median() for c in CFG10}).values * 100
a = mini(x + 3, 58, PW - 6, 22.5, 'Fig. 2'); a.imshow(M, cmap=CM, aspect='auto', vmin=0, vmax=12)
chip(x + PW / 2, 52.3, 'GBR · Ridge · MLP, raw / self-ref.')
chip(x + PW / 2, 48.0, 'tuned GBR · 1D-CNN · PINN')
chip(x + PW / 2, 43.7, 'leave-one-domain-out (13)')
chip(x + PW / 2, 39.4, 'leave-one-dataset-out (4)')

# ---- 3 monitoring: trajectory + LB scatter ----
x = xs[2]
P = pd.read_csv(f'{OUT}/pred_mlp_raw.csv.gz'); Cm = pd.read_csv(f'{OUT}/cells_mlp_raw.csv')
cand = Cm[(Cm.viol < 0.001) & (Cm.rmse < 0.08)]; ex = cand.sort_values('LB').iloc[int(0.85 * len(cand))].cell
g = P[P.cell == ex]
a = mini(x + 2.5, 69, PW - 5, 13.5, 'Fig. 3b')
a.plot(g.k, g.p, color=ORANGE, lw=0.6); a.plot(g.k, project(g.p.values), color=BLUE, lw=1.8); a.plot(g.k, g.soh, color=INK, lw=1.2, ls='--')
a = mini(x + 2.5, 50.5, 11.5, 13.5, 'Fig. 3c')
for c, col in zip(CFG, [BLUE, BLUE, ORANGE, ORANGE, GREEN, GREEN]):
    d = pd.read_csv(f'{OUT}/cells_{c}.csv'); a.scatter(d.rmse, d.LB, s=1.2, color=col, alpha=.5)
a.set_xscale('log'); a.set_yscale('log'); a.plot([1e-3, 3], [1e-3, 3], color=INK, lw=1); a.set_xlim(2e-3, 3); a.set_ylim(1e-4, 3)
a.set_xticks([]); a.set_yticks([]); a.minorticks_off()
S = pd.read_csv(f'{OUT}/tab_signals.csv')
a = mini(x + 15.6, 50.5, 11.5, 13.5, 'Table 5'); a.barh(range(len(S)), S.auc2, color=ORANGE); a.axvline(0.5, color=INK, lw=1); a.set_xlim(0.3, 1)
chip(x + PW / 2, 45.0, 'projection onto non-increasing C')
chip(x + PW / 2, 40.7, 'certificate LB = RMS(p − q)')
chip(x + PW / 2, 36.4, '+ 4 warning signals')

# ---- 4 calibration: coverage + checks ----
x = xs[3]
Q = pd.read_csv(f'{OUT}/tab_conformal_domains.csv'); pv = Q.pivot(index='dom', columns='variant', values='coverage').sort_values('split')
a = mini(x + 2.5, 66.5, PW - 5, 16, 'Fig. 5')
for v, col, mk in [('split', ORANGE, 'o'), ('weighted', PINK, 's'), ('hier', BLUE, '^')]: a.scatter(pv[v], range(len(pv)), s=7, color=col, marker=mk)
a.axvline(0.9, color=INK, lw=1.2, ls='--'); a.set_xlim(0, 1.05)
T = pd.read_csv(f'{OUT}/tab_checks_causal.csv'); base = T[T.K == 0]['median'].values[0]
a = mini(x + 2.5, 47.5, PW - 5, 14, 'Fig. 6')
for nm, col in [('fixed', BLUE), ('event', ORANGE), ('periodic_known', GREY)]:
    d = T[T.schedule == nm]; a.plot([0] + list(d.K), [base] + list(d['median']), color=col, lw=2, marker='o', ms=2.5, ls='--' if nm == 'periodic_known' else '-')
chip(x + PW / 2, 43.4, 'conformal 90%: split, weighted,')
chip(x + PW / 2, 39.6, 'group, hierarchical')
chip(x + PW / 2, 35.8, 'K = 1–5 causal checks', fs=10.2)

# ---- 5 trustworthiness: reliability, explainability, generalizability ----
x = xs[4]
R = pd.read_csv(f'{OUT}/rel_runs.csv')
a = mini(x + 2.5, 70.5, 11.5, 11.5, 'Fig. S1')
for i, cfg in enumerate(['gbr_self', 'mlp_raw']):
    d = R[R.cfg == cfg]; a.scatter(np.full(len(d), 2 * i), d.med_rmse, s=8, color=ORANGE); a.scatter(np.full(len(d), 2 * i + 0.8), d.med_rmse_q, s=8, color=BLUE, marker='s')
a.set_xlim(-0.6, 3.4)
I = pd.read_csv(f'{OUT}/xai_importance.csv', index_col=0); I = I[I.mean().sort_values(ascending=False).index]
a = mini(x + 15.6, 70.5, 11.5, 11.5, 'Fig. 7'); a.imshow(I.div(I.sum(axis=1), axis=0).values, cmap=CM, aspect='auto')
K = pd.read_csv(f'{OUT}/tab_ndomains.csv')
a = mini(x + 2.5, 53, PW - 5, 12, 'Fig. 8'); a.fill_between(K.k, K.q25, K.q75, color='#f6e3b4'); a.plot(K.k, K.med_rmse, color=AMBER, lw=2, marker='o', ms=3)
a.plot(K.k, K.med_rmse_q, color=GREEN, lw=2, marker='s', ms=3); a.set_xscale('log', base=2); a.set_xticks([]); a.minorticks_off()
chip(x + PW / 2, 46.8, 'Reliability: 5 seeds, bootstrap')
chip(x + PW / 2, 42.5, 'Explainability: SHAP vs oracle')
chip(x + PW / 2, 38.2, 'Generalizability: unseen data')

# ---- bottom: controls and findings ----
panel(1.5, 1.2, 49.5, 29.5, GREY, 'Controls')
I4 = pd.read_csv(f'{OUT}/tab_gmp_injection.csv'); I0 = I4[I4.phi == 0]
a = mini(4, 4.6, 21, 15.5, 'Fig. 4a')
for b, col in [(0, BLUE), (0.02, ORANGE), (0.05, GREEN)]:
    d = I0[I0.bias == b]; a.plot(d.noise_sd, d.rmse, color=col, lw=2); a.plot(d.noise_sd, d.LB, color=col, lw=2, ls=':')
NA = pd.read_csv(f'{OUT}/tab_nasa.csv')
a = mini(28, 4.6, 20.5, 15.5, 'Table S2'); xx = np.arange(len(NA)); a.bar(xx - 0.2, NA.rmse, 0.4, color=ORANGE); a.bar(xx + 0.2, NA.rmse_q, 0.4, color=BLUE)
ax.text(14.5, 22.4, 'GMP: simulated truth', ha='center', fontsize=10.8, color=NAVY)
ax.text(38.25, 22.4, 'NASA: regeneration', ha='center', fontsize=10.8, color=NAVY)
arrow((26.2, 32.6), (26.2, 31.0), c=GREY)

panel(53, 1.2, 105.5, 29.5, BLUE, 'What a battery operator learns: question \u2192 section \u2192 answer')
_S = pd.read_csv(f'{OUT}/tab_stats_offline.csv')   # median share of squared error seen, range over the 11 configurations (Table 4)
SEEN = f'{100 * _S.removed_cell.min():.1f}\u2013{100 * _S.removed_cell.max():.1f}%'
F = [('Q1', 'Error on a new chemistry or protocol?', 'Sec. 3.1', 'median 2.4\u20138.0% SOH; domain > model'),
     ('Q2', 'Can the error be bounded without labels?', 'Sec. 3.2', f'LB \u2264 RMSE in 4,257/4,257; sees {SEEN}'),
     ('Q3', 'Can risky cells be flagged in advance?', 'Sec. 3.3', 'domains yes; cells no (\u03c1 \u2264 0.33)'),
     ('Q4', 'How many capacity checks are needed?', 'Sec. 3.4', '3 checks: 2.41 \u2192 1.86% SOH'),
     ('Q5', 'Are the findings trustworthy?', 'Sec. 3.5\u20133.7', 'seeds \u00b10.08; W = 0.74; unseen datasets')]
for i_, (o, t, st, m) in enumerate(F):
    yy = 22.2 - i_ * 4.3
    ax.text(55.2, yy, o, ha='left', va='center', fontsize=11.5, color=INK, bbox=dict(boxstyle='round,pad=0.25', fc='#dde8f8', ec='none'))
    ax.text(61.2, yy, t, ha='left', va='center', fontsize=11, color=INK)
    ax.text(101.5, yy, st, ha='left', va='center', fontsize=10.5, color=NAVY)
    ax.text(115.5, yy, m, ha='left', va='center', fontsize=10, color=NAVY)
arrow((118, 32.6), (118, 31.0))
fig.savefig(f'{FIG}/Figure_1.png', facecolor='white'); print('methodology figure done')

In [ ]:
%%writefile 09_reliability.py
"""Step 9. Reliability of the findings.
(a) Run-to-run stability: the leave-one-domain-out experiment is repeated with 5 random seeds and
    5 different training subsamples (offset of the every-6th-cycle stride) for the main configuration
    (GBR, self-referenced) and the noisiest configuration (MLP, raw). Headline quantities are recomputed.
(b) Statistical uncertainty: domain-cluster bootstrap (B = 2000) 95% CIs for the headline quantities
    of the reported run (outputs/cells_*.csv)."""
import warnings, numpy as np, pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from common import load_cells, certificate, OUT, SEED, QUICK
warnings.filterwarnings('ignore')
D, F = load_cells(); doms = sorted(D.dom.unique())
RUNS = [(s, s) for s in range(5)] if not QUICK else [(0, 0), (1, 1)]


def fold(tgt, model, ref, seed, off):
    X = F if ref == 'raw' else ['d_' + c for c in F]
    tr = D[D.dom != tgt].iloc[off::6]; te = D[D.dom == tgt]
    mu, sd = tr[X].mean(), tr[X].std() + 1e-12
    m = (HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, random_state=seed) if model == 'gbr'
         else MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=300, early_stopping=True, n_iter_no_change=10, random_state=seed))
    m.fit((tr[X] - mu) / sd, tr.soh)
    te = te.assign(p=m.predict((te[X] - mu) / sd)); rows = []
    for c, g in te.groupby('cell', sort=False):
        ce = certificate(g.p.values, g.soh.values)
        rows.append(dict(cell=c, dom=tgt, rmse=ce['rmse'], rmse_q=ce['rmse_q'], LB=ce['LB']))
    return rows


def headline(C):
    return dict(med_rmse=C.rmse.median(), med_rmse_q=C.rmse_q.median(), improved=(C.rmse_q <= C.rmse + 1e-12).mean(),
                lb_valid=(C.LB <= C.rmse + 1e-12).mean(), removed=np.median(C.LB ** 2 / C.rmse ** 2),
                rel_red=np.median(1 - C.rmse_q / C.rmse))


rows = []
for model, ref in [('gbr', 'self'), ('mlp', 'raw')]:
    for seed, off in RUNS:
        C = pd.DataFrame([r for o in Parallel(n_jobs=2)(delayed(fold)(t, model, ref, seed, off) for t in doms) for r in o])
        rows.append(dict(cfg=f'{model}_{ref}', seed=seed, offset=off, **headline(C)))
        print(rows[-1], flush=True)
R = pd.DataFrame(rows); R.to_csv(f'{OUT}/rel_runs.csv', index=False)
S = R.groupby('cfg')[['med_rmse', 'med_rmse_q', 'improved', 'lb_valid', 'removed', 'rel_red']].agg(['mean', 'std', 'min', 'max'])
S.to_csv(f'{OUT}/tab_reliability_runs.csv'); print(S.round(4).T.to_string())

# (b) domain-cluster bootstrap of the reported run
rng = np.random.default_rng(SEED); out = []
for cfg in ['gbr_self', 'gbr_raw', 'mlp_self', 'mlp_raw', 'ridge_self', 'ridge_raw']:
    try: C = pd.read_csv(f'{OUT}/cells_{cfg}.csv')
    except FileNotFoundError: continue
    est = headline(C); dd = C.dom.unique(); bt = []
    for _ in range(2000):
        pick = rng.choice(dd, len(dd), replace=True); bt.append(headline(pd.concat([C[C.dom == d] for d in pick])))
    bt = pd.DataFrame(bt)
    for k, v in est.items():
        lo, hi = np.percentile(bt[k], [2.5, 97.5]); out.append(dict(cfg=cfg, quantity=k, estimate=v, lo=lo, hi=hi))
B = pd.DataFrame(out); B.to_csv(f'{OUT}/tab_reliability_ci.csv', index=False)
print(B.pivot(index='cfg', columns='quantity', values='estimate').round(4))

In [ ]:
%%writefile 10_explain.py
"""Step 10. Explainability of the SOH estimator and of its cross-domain error (TreeSHAP).
For each leave-one-domain-out fold (main configuration: GBR, self-referenced features):
  (a) global importance = mean |SHAP| on the held-out domain; stability of the feature ranking across
      the 13 fold models (Kendall's W, mean pairwise Spearman).
  (b) bias attribution: by SHAP additivity, mean prediction on target minus mean prediction on source
      equals the sum over features of (mean SHAP on target - mean SHAP on source). Each feature's term is
      its contribution to the domain-level shift of the prediction. We compare the total shift with the
      domain's measured bias and identify which features carry it.
  (c) error-consistency: correlation between |SHAP shift| of a feature and the feature's standardised
      mean difference between source and target (does the explanation point to the shifted inputs?)."""
import warnings, numpy as np, pandas as pd, shap
from itertools import combinations
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from common import load_cells, OUT, SEED, QUICK
warnings.filterwarnings('ignore')
D, F = load_cells(); X = ['d_' + c for c in F]; doms = sorted(D.dom.unique())
NS = 600 if QUICK else 2000
rng = np.random.default_rng(SEED)
imp, att, rows = {}, {}, []
for tgt in doms:
    tr = D[D.dom != tgt].iloc[::6]; te = D[D.dom == tgt]
    mu, sd = tr[X].mean(), tr[X].std() + 1e-12
    Ztr, Zte = (tr[X] - mu) / sd, (te[X] - mu) / sd
    m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, random_state=SEED).fit(Ztr, tr.soh)
    ex = shap.TreeExplainer(m)
    s_te = ex.shap_values(Zte.iloc[rng.choice(len(Zte), min(NS, len(Zte)), replace=False)])
    s_tr = ex.shap_values(Ztr.iloc[rng.choice(len(Ztr), NS, replace=False)])
    imp[tgt] = np.abs(s_te).mean(0)
    shift = s_te.mean(0) - s_tr.mean(0); att[tgt] = shift
    p_te = m.predict(Zte); bias = float(np.mean(p_te - te.soh.values))
    smd = (te[X].mean() - tr[X].mean()) / (tr[X].std() + 1e-12)
    rows.append(dict(dom=tgt, bias=bias, pred_shift=float(p_te.mean() - m.predict(Ztr).mean()), shap_shift_sum=float(shift.sum()),
                     true_shift=float(te.soh.mean() - tr.soh.mean()),
                     top_feature=F[int(np.argmax(np.abs(shift)))], top_share=float(np.abs(shift).max() / np.abs(shift).sum()),
                     rho_shift_smd=spearmanr(np.abs(shift), np.abs(smd.values)).correlation))
    print(rows[-1], flush=True)
I = pd.DataFrame(imp, index=F).T; I.to_csv(f'{OUT}/xai_importance.csv')
A = pd.DataFrame(att, index=F).T; A.to_csv(f'{OUT}/xai_shift.csv')
R = pd.DataFrame(rows); R.to_csv(f'{OUT}/xai_domains.csv', index=False)
# ranking stability across fold models
ranks = I.rank(axis=1, ascending=False).values; m_, n_ = ranks.shape
Rj = ranks.sum(0); W = 12 * np.sum((Rj - Rj.mean()) ** 2) / (m_ ** 2 * (n_ ** 3 - n_))
pair = [spearmanr(I.iloc[a], I.iloc[b]).correlation for a, b in combinations(range(m_), 2)]
top = I.mean().sort_values(ascending=False)
# share of total importance in top 3 features; agreement of top-3 sets across folds
top3 = [set(I.columns[np.argsort(-I.iloc[i].values)[:3]]) for i in range(m_)]
jac = np.mean([len(a & b) / len(a | b) for a, b in combinations(top3, 2)])
summ = dict(kendall_W=W, mean_pairwise_spearman=float(np.mean(pair)), min_pairwise_spearman=float(np.min(pair)),
            top3_jaccard=float(jac), top_features=';'.join(top.index[:5]),
            rho_bias_vs_predshift=spearmanr(R.bias, R.pred_shift - R.true_shift).correlation,
            median_rho_shift_smd=float(R.rho_shift_smd.median()))
pd.DataFrame([summ]).to_csv(f'{OUT}/tab_xai.csv', index=False)
print(pd.Series(summ)); print(top.round(4).to_string()); print(R[['dom', 'bias', 'top_feature', 'top_share', 'rho_shift_smd']].round(3).to_string(index=False))

In [ ]:
%%writefile 11_generalize.py
"""Step 11. Generalizability.
(a) Leave-one-dataset-out (LODSO): train on three of the four datasets (XJTU, TJU, HUST, MIT) and test on
    all cells of the fourth - a harder shift than leave-one-domain-out (new laboratory and protocol family).
(b) Training-domain diversity: for every target domain, train on k randomly chosen other domains
    (k = 1, 2, 4, 8, 12; 3 draws) and record unseen-domain error and the certificate quantities.
Main configuration (GBR) with self-referenced and raw features."""
import warnings, numpy as np, pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingRegressor
from common import load_cells, certificate, OUT, SEED, QUICK
warnings.filterwarnings('ignore')
D, F = load_cells(); doms = sorted(D.dom.unique())


def run(train_mask, test_mask, ref, seed=SEED):
    X = F if ref == 'raw' else ['d_' + c for c in F]
    tr = D[train_mask].iloc[::6]; te = D[test_mask]
    mu, sd = tr[X].mean(), tr[X].std() + 1e-12
    m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, random_state=seed).fit((tr[X] - mu) / sd, tr.soh)
    te = te.assign(p=m.predict((te[X] - mu) / sd)); rows = []
    for c, g in te.groupby('cell', sort=False):
        ce = certificate(g.p.values, g.soh.values)
        rows.append(dict(cell=c, dom=g.dom.iloc[0], dataset=g.dataset.iloc[0], rmse=ce['rmse'], rmse_q=ce['rmse_q'], LB=ce['LB']))
    return rows


# (a) leave-one-dataset-out
out = []
for ref in ['self', 'raw']:
    res = Parallel(n_jobs=2)(delayed(run)(D.dataset != ds, D.dataset == ds, ref) for ds in sorted(D.dataset.unique()))
    for r in res:
        for x in r: out.append(dict(ref=ref, **x))
L = pd.DataFrame(out); L.to_csv(f'{OUT}/gen_lodso_cells.csv', index=False)
LS = L.groupby(['ref', 'dataset']).agg(cells=('cell', 'size'), med_rmse=('rmse', 'median'), med_rmse_q=('rmse_q', 'median'),
                                       improved=('rmse_q', lambda s: np.nan), lb_valid=('LB', lambda s: np.nan)).reset_index()
LS['improved'] = L.assign(i=L.rmse_q <= L.rmse + 1e-12).groupby(['ref', 'dataset']).i.mean().values
LS['lb_valid'] = L.assign(i=L.LB <= L.rmse + 1e-12).groupby(['ref', 'dataset']).i.mean().values
# compare with leave-one-domain-out on the same cells
for ref in ['self', 'raw']:
    lodo = pd.read_csv(f'{OUT}/cells_gbr_{ref}.csv')
    for ds in LS[LS.ref == ref].dataset:
        LS.loc[(LS.ref == ref) & (LS.dataset == ds), 'med_rmse_lodo'] = lodo[lodo.dataset == ds].rmse.median()
LS.to_csv(f'{OUT}/tab_lodso.csv', index=False); print(LS.round(4).to_string(index=False))

# (b) number of training domains
KS = [1, 2, 4, 8, 12]; DRAWS = 1 if QUICK else 3; rng = np.random.default_rng(SEED); jobs = []
for tgt in doms:
    others = [d for d in doms if d != tgt]
    for k in KS:
        for r in range(DRAWS if k < 12 else 1):
            src = list(rng.choice(others, k, replace=False)); jobs.append((tgt, k, r, src))
def job(tgt, k, r, src):
    rows = run(D.dom.isin(src), D.dom == tgt, 'self')
    C = pd.DataFrame(rows)
    return dict(target=tgt, k=k, draw=r, med_rmse=C.rmse.median(), med_rmse_q=C.rmse_q.median(),
                lb_valid=(C.LB <= C.rmse + 1e-12).mean(), removed=np.median(C.LB ** 2 / C.rmse ** 2))
K = pd.DataFrame(Parallel(n_jobs=2)(delayed(job)(*j) for j in jobs)); K.to_csv(f'{OUT}/gen_ndomains.csv', index=False)
KS_ = K.groupby('k').agg(med_rmse=('med_rmse', 'median'), q25=('med_rmse', lambda s: s.quantile(.25)), q75=('med_rmse', lambda s: s.quantile(.75)),
                         med_rmse_q=('med_rmse_q', 'median'), lb_valid=('lb_valid', 'mean'), removed=('removed', 'median')).reset_index()
KS_.to_csv(f'{OUT}/tab_ndomains.csv', index=False); print(KS_.round(4).to_string(index=False))

In [ ]:
%%writefile 12_figures_trends.py
"""Step 12. Explainability, generalizability and reliability figures, saved under the Next Energy article numbering:
Figure_7 (explainability), Figure_8 (generalizability), Figure_S1 (reliability)."""
import numpy as np, pandas as pd, matplotlib, textwrap
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from common import OUT, FIG
plt.rcParams.update({'font.size': 13, 'font.weight': 'bold', 'axes.labelweight': 'bold', 'axes.titleweight': 'bold',
                     'axes.titlesize': 14, 'axes.labelsize': 13, 'xtick.labelsize': 12, 'ytick.labelsize': 12,
                     'legend.fontsize': 11, 'axes.spines.top': False, 'axes.spines.right': False, 'savefig.dpi': 300})
BLUE, ORANGE, GREEN, AMBER, PINK, GREY, INK = '#1f5fbf', '#d9541e', '#138a5e', '#c98a00', '#c2477d', '#8a8a85', '#111111'
def title(a, t):
    w = a.get_position().width * a.figure.get_figwidth()
    a.set_title(textwrap.fill(t, max(20, int(w * 72 / (14 * 0.6)))), loc='left', pad=8)
LAB = {'gbr_self': 'GBR, self-ref.', 'gbr_raw': 'GBR, raw', 'mlp_self': 'MLP, self-ref.', 'mlp_raw': 'MLP, raw',
       'ridge_self': 'Ridge, self-ref.', 'ridge_raw': 'Ridge, raw'}

# ---------- Fig. S1 (Supplementary Material): reliability ----------
R = pd.read_csv(f'{OUT}/rel_runs.csv'); B = pd.read_csv(f'{OUT}/tab_reliability_ci.csv')
fig, ax = plt.subplots(1, 2, figsize=(15, 5.6))
a = ax[0]
for i, cfg in enumerate(['gbr_self', 'mlp_raw']):
    d = R[R.cfg == cfg]
    a.scatter(np.full(len(d), 2 * i) + np.linspace(-.12, .12, len(d)), d.med_rmse * 100, s=90, color=ORANGE, label='before projection' if i == 0 else None)
    a.scatter(np.full(len(d), 2 * i + 0.8) + np.linspace(-.12, .12, len(d)), d.med_rmse_q * 100, s=90, color=BLUE, marker='s', label='after projection' if i == 0 else None)
a.set_xticks([0.4, 2.4]); a.set_xticklabels([LAB['gbr_self'], LAB['mlp_raw']]); a.set_ylabel('median per-cell RMSE (% SOH)')
a.legend(frameon=False); a.set_xlim(-0.6, 3.4)
title(a, '(a) Five seeds and training subsamples: projection removes most run-to-run variation of the noisy learner')
a = ax[1]; d = B[B.quantity == 'rel_red'].reset_index(drop=True); yy = np.arange(len(d))[::-1]
a.errorbar(d.estimate * 100, yy, xerr=[(d.estimate - d.lo) * 100, (d.hi - d.estimate) * 100], fmt='o', color=BLUE, ms=10, lw=3, capsize=6)
a.axvline(0, color=INK, ls='--', lw=2); a.set_yticks(yy); a.set_yticklabels([LAB[c] for c in d.cfg])
a.set_xlabel('median relative RMSE reduction by projection (%)')
title(a, '(b) The gain is positive for every configuration (95% domain-bootstrap CIs)')
fig.tight_layout(); fig.savefig(f'{FIG}/Figure_S1.png'); plt.close(fig)

# ---------- Fig. 7: explainability ----------
I = pd.read_csv(f'{OUT}/xai_importance.csv', index_col=0); A = pd.read_csv(f'{OUT}/xai_shift.csv', index_col=0)
Xd = pd.read_csv(f'{OUT}/xai_domains.csv').set_index('dom')
order = I.mean().sort_values(ascending=False).index
In = I[order].div(I.sum(axis=1), axis=0) * 100
In = pd.concat([In.iloc[:, :8], In.iloc[:, 8:].sum(axis=1).rename('other 8 features')], axis=1)
fig, ax = plt.subplots(1, 2, figsize=(17, 6.6), gridspec_kw=dict(width_ratios=[1.2, 1]))
a = ax[0]
from matplotlib.colors import LinearSegmentedColormap
cm = LinearSegmentedColormap.from_list('l', plt.get_cmap('YlOrBr')(np.linspace(0, .6, 256)))
im = a.imshow(In.values, cmap=cm, aspect='auto', vmin=0, vmax=45)
for i in range(In.shape[0]):
    for j in range(In.shape[1]):
        a.text(j, i, f'{In.values[i, j]:.0f}', ha='center', va='center', fontsize=11, color=INK)
a.set_xticks(range(In.shape[1])); a.set_xticklabels(In.columns, rotation=45, ha='right', fontsize=11)
a.set_yticks(range(len(In))); a.set_yticklabels(In.index, fontsize=11)
cb = fig.colorbar(im, ax=a, fraction=0.035); cb.set_label('share of mean |SHAP| (%)')
title(a, '(a) Feature importance on each unseen domain: the two charged-capacity features dominate in all 13 models')
a = ax[1]
E = pd.read_csv(f'{OUT}/xai_error_domains.csv').set_index('dom').loc[In.index]; T_ = pd.read_csv(f'{OUT}/xai_error_terms.csv', index_col=0).loc[In.index]
yy = np.arange(len(E))[::-1]; h = 0.36
a.barh(yy + h / 2, E.base_term * 100, h, color=GREY, label='expected-output difference (label level)')
a.barh(yy - h / 2, E.feature_terms * 100, h, color=BLUE, label='feature-use difference (sum over features)')
a.scatter(E.gap_src_minus_oracle * 100, yy, color=ORANGE, marker='*', s=240, zorder=5, label='deployed minus oracle prediction')
a.axvline(0, color=INK, lw=1.5); a.set_yticks(yy); a.set_yticklabels(E.index, fontsize=11)
a.set_xlabel('contribution to the mean gap from a target-trained oracle (% SOH)')
a.legend(frameon=False, fontsize=10.5, loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=1)
title(a, '(b) Error-centred decomposition: deployed model versus a model trained on the target domain')
fig.tight_layout(); fig.savefig(f'{FIG}/Figure_7.png'); plt.close(fig)

# ---------- Fig. 8: generalizability ----------
L = pd.read_csv(f'{OUT}/tab_lodso.csv'); K = pd.read_csv(f'{OUT}/tab_ndomains.csv'); KK = pd.read_csv(f'{OUT}/gen_ndomains.csv')
fig, ax = plt.subplots(1, 3, figsize=(18, 5.6))
a = ax[0]; ds = sorted(L.dataset.unique()); x = np.arange(len(ds)); w = 0.2
for i, (ref, col_lodo, col_lods) in enumerate([('self', '#a9c6f0', BLUE), ('raw', '#f5ae8c', ORANGE)]):
    d = L[L.ref == ref].set_index('dataset').loc[ds]
    a.bar(x + (2 * i - 1.5) * w, d.med_rmse_lodo * 100, w, color=col_lodo, edgecolor=col_lods, lw=2, label=f'{ref}: leave-one-domain-out')
    a.bar(x + (2 * i - 0.5) * w, d.med_rmse * 100, w, color=col_lods, label=f'{ref}: leave-one-dataset-out')
a.set_xticks(x); a.set_xticklabels(ds); a.set_ylabel('median per-cell RMSE (% SOH)'); a.legend(frameon=False, fontsize=10)
title(a, '(a) Removing a whole dataset raises error, most for MIT')
a = ax[1]
a.fill_between(K.k, K.q25 * 100, K.q75 * 100, color='#a9c6f0', alpha=.6, label='interquartile range over targets')
a.plot(K.k, K.med_rmse * 100, color=BLUE, marker='o', lw=3, ms=9, label='median, before projection')
a.plot(K.k, K.med_rmse_q * 100, color=GREEN, marker='s', lw=3, ms=9, label='median, after projection')
a.set_xscale('log', base=2); a.set_xticks(K.k); a.set_xticklabels(K.k); a.set_xlabel('number of training domains'); a.set_ylabel('median per-cell RMSE (% SOH)')
a.legend(frameon=False, fontsize=10.5)
title(a, '(b) Error falls as more domains are used for training')
a = ax[2]
a.plot(K.k, K.removed * 100, color=ORANGE, marker='^', lw=3, ms=10)
a.set_xscale('log', base=2); a.set_xticks(K.k); a.set_xticklabels(K.k); a.set_xlabel('number of training domains'); a.set_ylabel('squared error removed by projection (%)')
title(a, '(c) As bias shrinks, a larger share of error becomes visible to the certificate')
fig.tight_layout(); fig.savefig(f'{FIG}/Figure_8.png'); plt.close(fig)
print('trend figures done')

In [ ]:
%%writefile 13_tables.py
"""Step 13. Export every data table of the Next Energy article as CSV files and one HTML page, under the article's
numbering: main Tables 4, 5 and 6 and Supplementary Tables S1-S14. Text-only tables (1-3 and 7) are in the manuscript."""
import os, numpy as np, pandas as pd
from common import OUT
T = os.path.join(OUT, 'tables'); os.makedirs(T, exist_ok=True)
for f in os.listdir(T): os.remove(os.path.join(T, f))
pc = lambda x: np.round(100 * np.asarray(x, float), 2)
LAB = {'gbr_self': 'GBR, self-ref.', 'gbr_raw': 'GBR, raw', 'mlp_self': 'MLP, self-ref.', 'mlp_raw': 'MLP, raw',
       'ridge_self': 'Ridge, self-ref.', 'ridge_raw': 'Ridge, raw', 'gbrtuned_self': 'Tuned GBR, self-ref.',
       'gbrfull_self': 'GBR, self-ref., all cycles', 'cnn_self': '1D-CNN, self-ref.', 'pinn': 'PINN (Wang et al., 2024)', 'pinnpub': 'PINN, published per-cell scaling (non-deployable)'}
fp = lambda p: '< 0.001' if p < 0.001 else f'{p:.3f}'
tabs = {}

# ---- Table 4: estimators and certificate, with clustering-aware inference
S = pd.read_csv(f'{OUT}/tab_stats_offline.csv'); O = pd.read_csv(f'{OUT}/tab_certificate_offline.csv').set_index('cfg')
tabs['Table4_estimators_certificate'] = pd.DataFrame({
    'Configuration': S.cfg.map(LAB),
    'Median RMSE, cells / domains (%)': [f'{100*a:.2f} / {100*b:.2f}' for a, b in zip(S.med_rmse_cell, S.med_rmse_domainw)],
    'After projection, cells / domains (%)': [f'{100*a:.2f} / {100*b:.2f}' for a, b in zip(S.med_rmse_q_cell, S.med_rmse_q_domainw)],
    'Cells improved (%)': pc(O.loc[S.cfg].improved), 'Domains improved (of 13)': S.domains_improved,
    'LB <= RMSE, % (95% CI)': [f'{100*a:.1f} ({100*b:.1f}-{100*c:.1f})' for a, b, c in zip(S.lb_valid, S.lb_valid_lo, S.lb_valid_hi)],
    'Squared error removed, median % (95% CI)': [f'{100*a:.1f} ({100*b:.1f}-{100*c:.1f})' for a, b, c in zip(S.removed_cell, S.removed_lo, S.removed_hi)],
    'p, domain-level Wilcoxon (Holm)': [fp(p) for p in S.p_domain_wilcoxon_holm], 'p, mixed model (Holm)': [fp(p) for p in S.p_mixed_holm]})

# ---- Table 5: label-free signals
s = pd.read_csv(f'{OUT}/tab_signals.csv'); f2 = lambda a, b, c: [f'{x:.2f} ({y:.2f}, {z:.2f})' for x, y, z in zip(a, b, c)]
tabs['Table5_signals'] = pd.DataFrame({'Signal': s.signal, 'rho pooled': f2(s.rho, s.rho_lo, s.rho_hi), 'rho within domain': f2(s.rho_within, s.rho_within_lo, s.rho_within_hi),
    'AUROC >2%': f2(s.auc2, s.auc2_lo, s.auc2_hi), 'AUROC >5%': f2(s.auc5, s.auc5_lo, s.auc5_hi)})

# ---- Table S12: conformal variants
c = pd.read_csv(f'{OUT}/tab_conformal.csv'); NM = {'split': 'Split (pooled cells)', 'weighted': 'Weighted (covariate shift)', 'family': 'Group (dataset family)', 'hier': 'Hierarchical (domain level)'}
tabs['TableS12_conformal'] = pd.DataFrame({'Calibration': c.variant.map(NM), 'Overall coverage (%)': pc(c.coverage), 'Mean domain coverage (%)': pc(c.domain_mean_coverage),
    'Lowest domain (%)': [f'{100*a:.0f} ({b})' for a, b in zip(c.min_domain, c.min_domain_name)], 'Domains below 80%': c.domains_below_80,
    'Median bound / RMSE': c.width_median.round(2), 'Infinite bounds (%)': pc(c.infinite_bounds)})

# ---- Table S13: causal reference-check schedules
k = pd.read_csv(f'{OUT}/tab_checks_causal.csv'); piv = k.pivot_table(index='K', columns='schedule', values='median')
ev = k[k.schedule == 'event'].set_index('K'); fx = k[k.schedule == 'fixed'].set_index('K')
tabs['TableS13_checks_causal'] = pd.DataFrame({'K': piv.index, 'Fixed interval (%)': pc(piv['fixed'].fillna(piv['none'])),
    'Event-triggered (%)': pc(piv['event'].fillna(piv['none'])), 'Checks used, event (mean)': ev.checks_used.reindex(piv.index).round(2).fillna(0),
    'Periodic, known life (reference, %)': pc(piv['periodic_known'].fillna(piv['none'])),
    'Fixed vs none: domains better, p (Holm)': [('-' if K == 0 else f'{int(fx.loc[K, "domains_better_than_none"])}/13, {fp(fx.loc[K, "p_vs_none_holm"])}') for K in piv.index],
    'Event vs fixed: domains better, p (Holm)': [('-' if K == 0 else f'{int(ev.loc[K, "domains_better_than_fixed"])}/13, {fp(ev.loc[K, "p_vs_fixed_holm"])}') for K in piv.index]}).reset_index(drop=True)

# ---- Table 6: explainability
x = pd.read_csv(f'{OUT}/tab_xai.csv').iloc[0]; xe = pd.read_csv(f'{OUT}/tab_xai_error.csv').iloc[0]
I = pd.read_csv(f'{OUT}/xai_importance.csv', index_col=0); sh = I.div(I.sum(axis=1), axis=0).mean()
E = pd.read_csv(f'{OUT}/xai_error_domains.csv')
tabs['Table6_explainability'] = pd.DataFrame({'Quantity': [
    "Kendall's W of feature ranking, 13 models (permutation null mean; p)", 'Mean pairwise Spearman of importance: leave-one-domain-out models',
    'Mean (range) Spearman of importance: models trained on disjoint halves of the domains', 'Mean importance share: CC Q / CV Q (%)',
    'Oracle RMSE on the target domain, median (%)', 'Share of the deployed-minus-oracle gap from the expected-output (label-level) term, median (%)',
    'Domains in which the top error feature equals the top prediction-shift feature', 'Spearman of error terms with prediction-shift terms, median'],
    'Value': [f'{x.kendall_W:.3f} ({xe.W_null_mean:.3f}; {fp(xe.W_null_p)})', f'{xe.rho_lodo_pairs_mean:.2f}',
              f'{xe.rho_disjoint_halves_mean:.2f} ({xe.rho_disjoint_halves_min:.2f}-{xe.rho_disjoint_halves_max:.2f})', f'{100*sh["CC Q"]:.1f} / {100*sh["CV Q"]:.1f}',
              f'{100*xe.median_oracle_rmse:.2f}', f'{100*xe.median_base_share:.0f}', f'{int(round(xe.top_feature_agreement*len(E)))} of {len(E)}', f'{xe.median_rho_error_vs_shift:.2f}']})

# ---- Table S14: generalizability
l = pd.read_csv(f'{OUT}/tab_lodso.csv'); kk = pd.read_csv(f'{OUT}/tab_ndomains.csv')
tabs['TableS14a_leave_one_dataset_out'] = pd.DataFrame({'Features': l.ref, 'Held-out dataset': l.dataset, 'Cells': l.cells,
    'Leave-one-domain-out (%)': pc(l.med_rmse_lodo), 'Leave-one-dataset-out (%)': pc(l.med_rmse), 'After projection (%)': pc(l.med_rmse_q),
    'Cells improved (%)': pc(l.improved), 'LB valid (%)': pc(l.lb_valid)})
tabs['TableS14b_training_domains'] = pd.DataFrame({'Training domains k': kk.k, 'Median of domain medians (%)': pc(kk.med_rmse), 'IQR (%)': [f'{100*a:.2f}-{100*b:.2f}' for a, b in zip(kk.q25, kk.q75)],
    'After projection (%)': pc(kk.med_rmse_q), 'LB valid (%)': pc(kk.lb_valid), 'Squared error removed (%)': pc(kk.removed)})

# ================= Supplementary =================
n = pd.read_csv(f'{OUT}/tab_certificate_online.csv'); so = pd.read_csv(f'{OUT}/tab_stats_online.csv').set_index('cfg').loc[n.cfg]
tabs['TableS1_online'] = pd.DataFrame({'Configuration': n.cfg.map(LAB), 'Median online RMSE (%)': pc(n.med_raw), 'After prefix projection (%)': pc(n.med_q),
    'Cells improved (%)': pc(n.improved), 'Domains improved (of 13)': so.domains_improved.values, 'Prefix LB holds (%)': pc(n.lb_holds),
    'p, domain-level Wilcoxon (Holm)': [fp(p) for p in so.p_domain_wilcoxon_holm], 'p, mixed model (Holm)': [fp(p) for p in so.p_mixed_holm]})
na = pd.read_csv(f'{OUT}/tab_nasa.csv')
tabs['TableS2_nasa'] = pd.DataFrame({'Learner': na.learner, 'Cell': na.cell, 'RMSE (%)': pc(na.rmse), 'After projection (%)': pc(na.rmse_q),
    'LB (%)': pc(na.LB), 'delta (%)': pc(na.viol), 'Projection helped': np.where(na.improved, 'Yes', 'No')})
r = pd.read_csv(f'{OUT}/rel_runs.csv'); g = r.groupby('cfg')
tabs['TableS3_reliability'] = pd.DataFrame({'Configuration': [LAB[k_] for k_ in g.groups],
    'Median RMSE, mean +/- SD (%)': [f'{100*v.med_rmse.mean():.2f} +/- {100*v.med_rmse.std():.2f}' for _, v in g],
    'After projection, mean +/- SD (%)': [f'{100*v.med_rmse_q.mean():.2f} +/- {100*v.med_rmse_q.std():.2f}' for _, v in g],
    'Cells improved, mean (%)': [pc(v.improved.mean()) for _, v in g], 'LB valid, mean (%)': [pc(v.lb_valid.mean()) for _, v in g],
    'Squared error removed, mean (%)': [pc(v.removed.mean()) for _, v in g]})
c9 = pd.read_csv(f'{OUT}/tab_checks.csv'); pv = c9.pivot_table(index='K', columns='placement', values='median').reindex(columns=['none', 'periodic', 'fade', 'departure'])
st = pd.read_csv(f'{OUT}/tab_stats_checks.csv'); dp = st[st.placement == 'departure'].set_index('K')
tabs['TableS4_checks_offline'] = pd.DataFrame({'K': pv.index, 'Periodic (%)': pc(pv.periodic.fillna(pv['none'])), 'Equal-fade (%)': pc(pv.fade), 'Departure (%)': pc(pv.departure),
    'Departure vs periodic: domains better, p (Holm)': [('-' if K_ == 0 else f'{int(dp.loc[K_, "domains_better"])}/13, {fp(dp.loc[K_, "p_domain_wilcoxon_holm"])}') for K_ in pv.index]}).reset_index(drop=True)
A = pd.read_csv(f'{OUT}/tab_alarm.csv')
tabs['TableS5_alarm'] = pd.DataFrame({'Mode': A['mode'], 'Threshold theta (%)': pc(A.theta), 'Event': A.event, 'Alarm rate (%)': pc(A.alarm_rate),
    'Prevalence (%)': pc(A.prevalence), 'Precision (%)': pc(A.precision), 'Recall (%)': pc(A.recall)})
sr = pd.read_csv(f'{OUT}/tab_soh_reference.csv'); md = pd.read_csv(f'{OUT}/tab_mlp_diag.csv')
tabs['TableS6_soh_reference'] = pd.DataFrame({'SOH reference': sr.soh_reference, 'Median RMSE (%)': pc(sr.med_rmse), 'After projection (%)': pc(sr.med_rmse_q),
    'Cells improved (%)': pc(sr.improved), 'LB valid (%)': pc(sr.lb_valid), 'Squared error removed (%)': pc(sr.removed), 'Largest initial SOH': sr.max_initial_soh.round(3)})
tabs['TableS7_mlp_training'] = md.groupby('cfg').agg(folds=('dom', 'size'), iterations_min=('n_iter', 'min'), iterations_median=('n_iter', 'median'),
    iterations_max=('n_iter', 'max'), convergence_warnings=('converged_warning', 'sum')).reset_index().rename(columns={'cfg': 'Configuration'})
hp = {}
for cfg, cols in [('gbrtuned_self', ['hp_learning_rate', 'hp_max_leaf_nodes', 'hp_max_iter', 'hp_l2_regularization']), ('cnn_self', ['hp_W', 'hp_lr', 'epochs']),
                  ('pinn', ['epochs']), ('pinnpub', ['epochs'])]:
    p_ = f'{OUT}/cells_{cfg}.csv'
    if os.path.exists(p_):
        hp[cfg] = pd.read_csv(p_).groupby('dom')[cols].first()
if hp:   # layout of Table S8: one row per held-out domain
    doms_ = sorted(set().union(*[set(v.index) for v in hp.values()]))
    g_ = lambda cfg, c_: hp[cfg][c_].reindex(doms_).values if cfg in hp else [None] * len(doms_)
    tabs['TableS8_selected_hyperparameters'] = pd.DataFrame({'Held-out domain': doms_,
        'Tuned GBR: learning rate': g_('gbrtuned_self', 'hp_learning_rate'), 'Tuned GBR: max. leaf nodes': g_('gbrtuned_self', 'hp_max_leaf_nodes'),
        'Tuned GBR: max. iterations': g_('gbrtuned_self', 'hp_max_iter'), 'Tuned GBR: L2': g_('gbrtuned_self', 'hp_l2_regularization'),
        '1D-CNN: W (cycles)': g_('cnn_self', 'hp_W'), '1D-CNN: learning rate': g_('cnn_self', 'hp_lr'), '1D-CNN: epochs': g_('cnn_self', 'epochs'),
        'PINN epochs: deployable': g_('pinn', 'epochs'), 'PINN epochs: published scaling': g_('pinnpub', 'epochs')})
pr = pd.read_csv(f'{OUT}/data_provenance.csv')
tabs['TableS9_data_provenance'] = pr[pr.removed > 0].rename(columns={'file': 'File', 'rows_raw': 'Rows in source', 'rows_kept': 'Rows kept', 'removed': 'Removed (missing/infinite)'})
cd_ = pd.read_csv(f'{OUT}/tab_conformal_domains.csv')   # layout of Table S10: coverage (Clopper-Pearson 95% interval) per domain and calibration
cd_['txt'] = [f'{100*a_:.2f} ({100*b_:.2f}-{100*c_:.2f})' for a_, b_, c_ in zip(cd_.coverage, cd_.lo, cd_.hi)]
w_ = cd_.pivot(index='dom', columns='variant', values='txt')
tabs['TableS10_conformal_domains'] = pd.DataFrame({'Domain': w_.index, 'Cells': cd_.groupby('dom').cells.first().reindex(w_.index).values,
    **{NM[v_]: w_[v_].values for v_ in ['split', 'weighted', 'family', 'hier'] if v_ in w_}})
tabs['TableS11_xai_error_domains'] = E.assign(**{c_: pc(E[c_]) for c_ in ['bias', 'oracle_rmse', 'gap_src_minus_oracle', 'base_term', 'feature_terms']}).round(3)

ORDER = ['Table4_estimators_certificate', 'Table5_signals', 'Table6_explainability'] + sorted(
    [k_ for k_ in tabs if k_.startswith('TableS')], key=lambda s_: (int(''.join(ch for ch in s_.split('_')[0][6:] if ch.isdigit())), s_))
tabs = {k_: tabs[k_] for k_ in ORDER}
html = ['<html><head><meta charset="utf-8"><style>body{font-family:Arial;}table{border-collapse:collapse;margin:12px 0 28px}td,th{border:1px solid #999;padding:4px 8px;font-size:13px}th{background:#e7eef7}</style></head><body>']
for name, df in tabs.items():
    df.to_csv(f'{T}/{name}.csv', index=False); html += [f'<h3>{name.replace("_", " ")}</h3>', df.to_html(index=False)]
open(f'{T}/all_tables.html', 'w').write('\n'.join(html + ['</body></html>'])); print('tables written:', list(tabs))

In [ ]:
%%writefile 14_strong.py
"""Step 14. Stronger estimators under the same leave-one-domain-out protocol (tests whether
the conclusions depend on estimator strength). Four additional configurations, all scored with the same certificate:
  gbrtuned_self : gradient boosting, hyperparameters chosen inside every outer fold by grouped
                  (domain-wise) 3-fold cross-validation on the 12 source domains (16-point grid).
  gbrfull_self  : the main GBR configuration trained on every cycle (stride 1) instead of every 6th.
  cnn_self      : causal 1D convolutional sequence model over a window of the last W cycles of
                  self-referenced features; W and learning rate chosen on two held-out source domains.
  pinn          : JAX port of the physics-informed network of Wang et al. (2024, PINN4SOH: Solution_u
                  encoder/predictor, dynamical_F network, data + PDE + monotonic physics losses, same
                  widths, sine activations, dropout and loss weights). Deployable preprocessing: z-score
                  of the 16 features with source statistics and cycle index / 1000.
  pinnpub       : the same network with the published per-cell whole-life min-max scaling of features and cycle
                  index; it uses future cycles of the test cell and is reported only as a non-deployable reference.
Usage: python 14_strong.py [gbr|cnn|pinn|pinnpub|all]"""
import sys, time, warnings, numpy as np, pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from common import load_cells, cell_rows, OUT, SEED, QUICK
warnings.filterwarnings('ignore')
D, F = load_cells(); XS = ['d_' + c for c in F]; DOMS = sorted(D.dom.unique())
if QUICK: DOMS = DOMS[:3]
WHAT = sys.argv[1] if len(sys.argv) > 1 else 'all'


def save(cfg, res):
    pd.concat([r[0] for r in res]).to_csv(f'{OUT}/pred_{cfg}.csv.gz', index=False, float_format='%.6f')
    C = pd.DataFrame([x for r in res for x in r[1]]); C.to_csv(f'{OUT}/cells_{cfg}.csv', index=False)
    print(f'{cfg}: median RMSE {C.rmse.median():.4f} after projection {C.rmse_q.median():.4f} '
          f'LB<=RMSE {np.mean(C.LB <= C.rmse + 1e-12):.3f}', flush=True)


def zs(tr, te, X):
    mu, sd = tr[X].mean(), tr[X].std() + 1e-12
    return ((tr[X] - mu) / sd).values.astype('float32'), ((te[X] - mu) / sd).values.astype('float32')


# ---------------------------------------------------------------- tuned and full-data GBR
GRID = [dict(learning_rate=lr, max_leaf_nodes=ml, max_iter=mi, l2_regularization=l2)
        for lr in (0.05, 0.1) for ml in (15, 63) for mi in (200, 600) for l2 in (0.0, 1.0)]


def gbr_tuned(tgt):
    tr = D[D.dom != tgt]; te = D[D.dom == tgt]
    inner = tr.iloc[::12]; Zi, _ = zs(inner, inner, XS)
    cv = []
    for h in GRID:
        e = []
        for a, b in GroupKFold(3).split(Zi, groups=inner.dom):
            m = HistGradientBoostingRegressor(random_state=SEED, **h).fit(Zi[a], inner.soh.values[a])
            e.append(np.sqrt(np.mean((m.predict(Zi[b]) - inner.soh.values[b]) ** 2)))
        cv.append(np.mean(e))
    best = GRID[int(np.argmin(cv))]
    tr6 = tr.iloc[::6]; Ztr, Zte = zs(tr6, te, XS)
    p = HistGradientBoostingRegressor(random_state=SEED, **best).fit(Ztr, tr6.soh.values).predict(Zte)
    te = te[['cell', 'dataset', 'k', 'soh']].assign(p=p)
    return te[['cell', 'k', 'soh', 'p']], [dict(r, **{f'hp_{k}': v for k, v in best.items()}) for r in cell_rows(te, tgt)]


def gbr_full(tgt):
    tr = D[D.dom != tgt]; te = D[D.dom == tgt]; Ztr, Zte = zs(tr, te, XS)
    p = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, random_state=SEED).fit(Ztr, tr.soh.values).predict(Zte)
    te = te[['cell', 'dataset', 'k', 'soh']].assign(p=p)
    return te[['cell', 'k', 'soh', 'p']], cell_rows(te, tgt)


# ---------------------------------------------------------------- JAX models
def jax_imports():
    global jax, jnp, optax
    import jax, jax.numpy as jnp, optax
    jax.config.update('jax_platform_name', 'cpu')


def dense_init(key, sizes):
    ps = []
    for i, (a, b) in enumerate(zip(sizes[:-1], sizes[1:])):
        k = jax.random.fold_in(key, i)
        ps.append((jax.random.normal(k, (a, b)) * np.sqrt(2.0 / (a + b)), jnp.zeros(b)))  # Xavier normal
    return ps


def windows(Z, cells, W):
    """Causal windows: for cycle k of a cell, rows max(0, k-W+1)..k, left-padded with the first cycle."""
    out = np.empty((len(Z), W, Z.shape[1]), 'float32'); start = 0
    for n in pd.Series(cells).groupby(cells, sort=False).size().values:
        z = Z[start:start + n]; pad = np.concatenate([np.repeat(z[:1], W - 1, 0), z])
        idx = np.arange(n)[:, None] + np.arange(W)[None, :]
        out[start:start + n] = pad[idx]; start += n
    return out


def cnn_fns():
    def init(key, d):
        k1, k2, k3 = jax.random.split(key, 3)
        return dict(c1=(jax.random.normal(k1, (5, d, 32)) * np.sqrt(2 / (5 * d)), jnp.zeros(32)),
                    c2=(jax.random.normal(k2, (5, 32, 32)) * np.sqrt(2 / (5 * 32)), jnp.zeros(32)),
                    out=dense_init(k3, [64, 32, 1]))

    def apply(P, x):  # x: (B, W, d)
        conv = lambda h, wb: jax.nn.relu(jax.lax.conv_general_dilated(h, wb[0], (1,), 'SAME', dimension_numbers=('NWC', 'WIO', 'NWC')) + wb[1])
        h = conv(conv(x, P['c1']), P['c2'])
        h = jnp.concatenate([h[:, -1, :], h.mean(1)], -1)  # last-step state and window mean
        (w1, b1), (w2, b2) = P['out']
        return (jax.nn.relu(h @ w1 + b1) @ w2 + b2)[:, 0]
    return init, apply


def train_nn(init, apply, Xtr, ytr, Xva, yva, lr, key, epochs=60, patience=8, bs=256):
    P = init(key, Xtr.shape[-1]); opt = optax.adam(lr); S = opt.init(P)
    loss = lambda P, x, y: jnp.mean((apply(P, x) - y) ** 2)

    @jax.jit
    def step(P, S, x, y):
        l, g = jax.value_and_grad(loss)(P, x, y); u, S = opt.update(g, S, P); return optax.apply_updates(P, u), S, l
    pred = jax.jit(apply)
    best, bestP, wait, rng = np.inf, P, 0, np.random.default_rng(SEED)
    for ep in range(epochs):
        perm = rng.permutation(len(Xtr))
        for i in range(0, len(perm) - bs + 1, bs):
            j = perm[i:i + bs]; P, S, _ = step(P, S, Xtr[j], ytr[j])
        v = float(np.sqrt(np.mean((np.concatenate([np.asarray(pred(P, Xva[i:i + 4096])) for i in range(0, len(Xva), 4096)]) - yva) ** 2)))
        if v < best - 1e-5: best, bestP, wait = v, P, 0
        else:
            wait += 1
            if wait >= patience: break
    return bestP, best, ep + 1


def predict_batched(apply, P, X):
    f = jax.jit(apply); return np.concatenate([np.asarray(f(P, X[i:i + 4096])) for i in range(0, len(X), 4096)])


def cnn_fold(tgt):
    init, apply = cnn_fns(); key = jax.random.PRNGKey(SEED)
    src = [d for d in DOMS if d != tgt] if not QUICK else [d for d in sorted(D.dom.unique()) if d != tgt]
    va_doms = list(np.random.default_rng(SEED + DOMS.index(tgt)).choice(src, 2, replace=False))
    tr = D[D.dom != tgt]; te = D[D.dom == tgt]
    mu, sd = tr[XS].mean(), tr[XS].std() + 1e-12
    Z = lambda d: ((d[XS] - mu) / sd).values.astype('float32')
    best = None
    for W in (10, 30):
        Xall = windows(Z(tr), tr.cell.values, W); yall = tr.soh.values.astype('float32')
        isva = tr.dom.isin(va_doms).values; sub = np.zeros(len(tr), bool); sub[::6] = True
        for lr in (1e-3, 3e-3):
            P, v, ep = train_nn(init, apply, Xall[sub & ~isva], yall[sub & ~isva], Xall[isva][::3], yall[isva][::3], lr, key)
            if best is None or v < best[0]: best = (v, W, lr, ep)
    v, W, lr, _ = best
    # refit on all source domains with the chosen W and lr; early stopping on the same validation split
    Xall = windows(Z(tr), tr.cell.values, W); yall = tr.soh.values.astype('float32')
    isva = tr.dom.isin(va_doms).values; sub = np.zeros(len(tr), bool); sub[::6] = True
    P, _, ep = train_nn(init, apply, Xall[sub & ~isva], yall[sub & ~isva], Xall[isva][::3], yall[isva][::3], lr, key)
    p = predict_batched(apply, P, windows(Z(te), te.cell.values, W))
    te = te[['cell', 'dataset', 'k', 'soh']].assign(p=p)
    return te[['cell', 'k', 'soh', 'p']], [dict(r, hp_W=W, hp_lr=lr, epochs=ep) for r in cell_rows(te, tgt)]


def pinn_fold(tgt, epochs=120, warm=15, patience=20, bs=256, norm='causal'):
    """JAX port of PINN4SOH (Wang et al., 2024). Architecture and loss as in Model/Model.py:
    u = Solution_u([x, t]) with encoder MLP(17-60-60-32, sine, dropout 0.2) and predictor (dropout 0.2, 32-32 sine, 32-1);
    F = MLP(35-60-60-1, sine, dropout 0.2) on [x, t, u, u_x, u_t]; f = u_t - F;
    loss = MSE(u, y) + 0.7 MSE(f, 0) + 0.2 sum relu((u2 - u1)(y1 - y2)) over consecutive-cycle pairs;
    Adam, warm-up then cosine learning rate for Solution_u (0.002 -> 0.01 -> 0.0002), lr_F = 0.001,
    early stopping on a random 20% of source cells. Epochs shortened from 200 to 120 for CPU time."""
    key = jax.random.PRNGKey(SEED)
    ku, kp, kf = jax.random.split(key, 3)
    P = dict(enc=dense_init(ku, [17, 60, 60, 32]), pre=dense_init(kp, [32, 32, 1]), F=dense_init(kf, [35, 60, 60, 1]))
    tr = D[D.dom != tgt]; te = D[D.dom == tgt]
    mu, sd = tr[F].mean(), tr[F].std() + 1e-12
    def X(d):
        if norm == 'causal':
            return np.column_stack([((d[F] - mu) / sd).values, d.k.values / 1000.0]).astype('float32')
        # published PINN4SOH preprocessing: per-cell min-max to [-1, 1] over the whole life (features and cycle index);
        # uses the test cell's future cycles, so it is reported only as a non-deployable reference
        Z = d[F + ['k']].astype(float); g = Z.groupby(d.cell.values, sort=False)
        Z = 2 * (Z - g.transform('min')) / (g.transform('max') - g.transform('min')).replace(0, np.nan) - 1
        return Z.fillna(0.0).values.astype('float32')
    rng = np.random.default_rng(SEED + 7)
    cells = tr.cell.unique(); va_cells = set(rng.choice(cells, int(0.2 * len(cells)), replace=False))
    isva = tr.cell.isin(va_cells).values
    Xtr, ytr = X(tr), tr.soh.values.astype('float32')
    nxt = np.r_[tr.cell.values[1:] == tr.cell.values[:-1], False]  # pair (k, k+1) within the same cell
    i1 = np.where(nxt & ~isva)[0][::6]
    Xva, yva = Xtr[isva][::3], ytr[isva][::3]

    def mlp(ps, h, keys, drop, rate=0.2, hidden_dropout=True):
        for i, (w, b) in enumerate(ps):
            h = h @ w + b
            if i < len(ps) - 1:
                h = jnp.sin(h)
                if drop and hidden_dropout and i == len(ps) - 2:
                    h = h * jax.random.bernoulli(keys[i], 1 - rate, h.shape) / (1 - rate)
        return h

    def u_single(P, x, t, key, drop):
        k = jax.random.split(key, 4)
        e = mlp(P['enc'], jnp.concatenate([x, t[None]]), k, drop)
        if drop: e = e * jax.random.bernoulli(k[3], 0.8, e.shape) / 0.8
        return mlp(P['pre'], e, k[2:], drop, hidden_dropout=False)[0]

    def forward(P, xt, key, drop=True):
        ks = jax.random.split(key, xt.shape[0] + 1)
        def one(x, t, k):
            u, (ux, ut) = jax.value_and_grad(u_single, argnums=(1, 2))(P, x, t, k, drop)
            return u, ux, ut
        u, ux, ut = jax.vmap(one)(xt[:, :-1], xt[:, -1], ks[1:])
        kF = jax.random.split(ks[0], 3)
        Fv = mlp(P['F'], jnp.concatenate([xt, u[:, None], ux, ut[:, None]], 1), kF, drop)[:, 0]
        return u, ut - Fv

    def loss(P, x1, x2, y1, y2, key):
        k1, k2 = jax.random.split(key)
        u1, f1 = forward(P, x1, k1); u2, f2 = forward(P, x2, k2)
        l1 = 0.5 * jnp.mean((u1 - y1) ** 2) + 0.5 * jnp.mean((u2 - y2) ** 2)
        l2 = 0.5 * jnp.mean(f1 ** 2) + 0.5 * jnp.mean(f2 ** 2)
        l3 = jnp.sum(jax.nn.relu((u2 - u1) * (y1 - y2)))
        return l1 + 0.7 * l2 + 0.2 * l3

    sched = np.concatenate([np.linspace(0.002, 0.01, warm), 0.0002 + 0.5 * (0.01 - 0.0002) * (1 + np.cos(np.pi * np.arange(epochs - warm) / (epochs - warm)))])
    optU = optax.inject_hyperparams(optax.adam)(learning_rate=0.002); optF = optax.adam(0.001)
    U = dict(enc=P['enc'], pre=P['pre']); SU, SF = optU.init(U), optF.init(P['F'])

    @jax.jit
    def step(U, Fp, SU, SF, x1, x2, y1, y2, key, lr):
        g = jax.grad(loss)(dict(U, F=Fp), x1, x2, y1, y2, key)
        SU.hyperparams['learning_rate'] = lr
        uu, SU = optU.update(dict(enc=g['enc'], pre=g['pre']), SU, U); U = optax.apply_updates(U, uu)
        uf, SF = optF.update(g['F'], SF, Fp); Fp = optax.apply_updates(Fp, uf)
        return U, Fp, SU, SF

    pred_u = jax.jit(jax.vmap(lambda Pp, x: u_single(Pp, x[:-1], x[-1], jax.random.PRNGKey(0), False), in_axes=(None, 0)))
    pv = lambda Pp, Xm: np.concatenate([np.asarray(pred_u(Pp, Xm[i:i + 8192])) for i in range(0, len(Xm), 8192)])
    Fp = P['F']; best, bestU, wait, rngb = np.inf, U, 0, np.random.default_rng(SEED)
    for ep in range(epochs):
        perm = rngb.permutation(i1)
        for b in range(0, len(perm) - bs + 1, bs):
            j = perm[b:b + bs]
            U, Fp, SU, SF = step(U, Fp, SU, SF, Xtr[j], Xtr[j + 1], ytr[j], ytr[j + 1], jax.random.PRNGKey(ep * 100000 + b), float(sched[ep]))
        v = float(np.mean((pv(U, Xva) - yva) ** 2))
        if v < best: best, bestU, wait = v, U, 0
        else:
            wait += 1
            if wait > patience: break
    p = pv(bestU, X(te))
    te = te[['cell', 'dataset', 'k', 'soh']].assign(p=p)
    return te[['cell', 'k', 'soh', 'p']], [dict(r, epochs=ep + 1) for r in cell_rows(te, tgt)]


if __name__ == '__main__':
    t0 = time.time()
    if WHAT in ('gbr', 'all'):
        save('gbrtuned_self', Parallel(n_jobs=2)(delayed(gbr_tuned)(t) for t in DOMS)); print('t', time.time() - t0, flush=True)
        save('gbrfull_self', Parallel(n_jobs=2)(delayed(gbr_full)(t) for t in DOMS)); print('t', time.time() - t0, flush=True)
    if WHAT in ('cnn', 'all'):
        jax_imports(); res = []
        for t in DOMS: res.append(cnn_fold(t)); print(t, res[-1][1][0].get('hp_W'), round(time.time() - t0), flush=True)
        save('cnn_self', res)
    if WHAT in ('pinn', 'all'):
        jax_imports(); res = []
        for t in DOMS: res.append(pinn_fold(t)); print(t, res[-1][1][0].get('epochs'), round(time.time() - t0), flush=True)
        save('pinn', res)
    if WHAT in ('pinnpub', 'all'):
        jax_imports(); res = []
        for t in DOMS: res.append(pinn_fold(t, norm='percell')); print(t, res[-1][1][0].get('epochs'), round(time.time() - t0), flush=True)
        save('pinnpub', res)

In [ ]:
%%writefile 15_stats.py
"""Step 15. Clustering-aware inference (cells are nested in 13 domains and share one fitted model per fold).
For every configuration:
  * domain-weighted summaries (mean of 13 domain medians) next to cell-pooled medians;
  * paired test of projection on the 13 domain medians (exact Wilcoxon signed-rank);
  * mixed-effects model on the per-cell log error ratio log(RMSE_q / RMSE) with a random intercept per domain;
  * domain-cluster bootstrap 95% CI of the median relative reduction and of the median removed share;
  * Clopper-Pearson 95% CI of the bound-validity rate per configuration (387 cells).
Holm correction within each family (offline projection, online projection, reference-check placements)."""
import glob, warnings, numpy as np, pandas as pd
from scipy.stats import wilcoxon, beta
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from common import OUT, SEED
warnings.filterwarnings('ignore')
CFGS = ['gbr_self', 'gbr_raw', 'ridge_self', 'ridge_raw', 'mlp_self', 'mlp_raw', 'gbrtuned_self', 'gbrfull_self', 'cnn_self', 'pinn', 'pinnpub']
CFGS = [c for c in CFGS if glob.glob(f'{OUT}/cells_{c}.csv')]
B = 2000; rng = np.random.default_rng(SEED)


def cp(k, n, a=0.05):
    return (beta.ppf(a / 2, k, n - k + 1) if k > 0 else 0.0, beta.ppf(1 - a / 2, k + 1, n - k) if k < n else 1.0)


def mixed(df, col):
    d = df.assign(r=np.log(df[col]))
    m = smf.mixedlm('r ~ 1', d, groups=d['dom']).fit(reml=True)
    return float(m.params['Intercept']), float(m.bse['Intercept']), float(m.pvalues['Intercept'])


def cluster_boot(df, stat):
    doms = df.dom.unique(); g = {d: df[df.dom == d] for d in doms}; out = []
    for _ in range(B):
        out.append(stat(pd.concat([g[d] for d in rng.choice(doms, len(doms))])))
    return np.percentile(out, [2.5, 97.5])


rows = []
for c in CFGS:
    C = pd.read_csv(f'{OUT}/cells_{c}.csv').assign(ratio=lambda d: d.rmse_q / d.rmse, removed=lambda d: d.LB ** 2 / d.rmse ** 2)
    dm = C.groupby('dom').agg(rmse=('rmse', 'median'), rmse_q=('rmse_q', 'median'), removed=('removed', 'median'))
    k = int((C.LB <= C.rmse + 1e-12).sum()); lo, hi = cp(k, len(C))
    est, se, p_mixed = mixed(C, 'ratio')
    ci_red = cluster_boot(C, lambda d: np.median(1 - d.ratio)); ci_rem = cluster_boot(C, lambda d: np.median(d.removed))
    rows.append(dict(cfg=c, cells=len(C), domains=C.dom.nunique(),
                     med_rmse_cell=C.rmse.median(), med_rmse_domainw=dm.rmse.mean(),
                     med_rmse_q_cell=C.rmse_q.median(), med_rmse_q_domainw=dm.rmse_q.mean(),
                     removed_cell=C.removed.median(), removed_domainw=dm.removed.mean(), removed_lo=ci_rem[0], removed_hi=ci_rem[1],
                     red_median=float(np.median(1 - C.ratio)), red_lo=ci_red[0], red_hi=ci_red[1],
                     domains_improved=int((dm.rmse_q < dm.rmse).sum()),
                     p_domain_wilcoxon=float(wilcoxon(dm.rmse_q, dm.rmse, method='exact').pvalue),
                     mixed_log_ratio=est, mixed_se=se, p_mixed=p_mixed,
                     lb_valid=k / len(C), lb_valid_lo=lo, lb_valid_hi=hi))
T = pd.DataFrame(rows)
for col in ['p_domain_wilcoxon', 'p_mixed']: T[col + '_holm'] = multipletests(T[col], method='holm')[1]
T.to_csv(f'{OUT}/tab_stats_offline.csv', index=False); print(T.round(4).to_string(index=False))

# online, domain level (cells_online.csv from step 2, which includes every configuration present)
O = pd.read_csv(f'{OUT}/cells_online.csv'); O['dom'] = O.cell.str.split('/').str[0]
on = []
for c, g in O.groupby('cfg'):
    dm = g.groupby('dom')[['rmse_online', 'rmse_online_q']].median()
    est, se, p = mixed(g.assign(ratio=g.rmse_online_q / g.rmse_online), 'ratio')
    on.append(dict(cfg=c, med_online_cell=g.rmse_online.median(), med_online_q_cell=g.rmse_online_q.median(),
                   med_online_domainw=dm.rmse_online.mean(), med_online_q_domainw=dm.rmse_online_q.mean(),
                   domains_improved=int((dm.rmse_online_q < dm.rmse_online).sum()),
                   p_domain_wilcoxon=float(wilcoxon(dm.rmse_online_q, dm.rmse_online, method='exact').pvalue),
                   mixed_log_ratio=est, p_mixed=p))
O2 = pd.DataFrame(on)
for col in ['p_domain_wilcoxon', 'p_mixed']: O2[col + '_holm'] = multipletests(O2[col], method='holm')[1]
O2.to_csv(f'{OUT}/tab_stats_online.csv', index=False); print(O2.round(4).to_string(index=False))

# reference-check placements, domain level, Holm over K x placement
R = pd.read_csv(f'{OUT}/cells_checks.csv'); ck = []
for K in range(1, 6):
    for nm in ['fade', 'departure']:
        dm = R.groupby('dom')[[f'{nm}{K}', f'periodic{K}']].median()
        est, se, p = mixed(R.assign(ratio=R[f'{nm}{K}'] / R[f'periodic{K}']), 'ratio')
        ck.append(dict(K=K, placement=nm, domains_better=int((dm[f'{nm}{K}'] < dm[f'periodic{K}']).sum()),
                       p_domain_wilcoxon=float(wilcoxon(dm[f'{nm}{K}'], dm[f'periodic{K}'], method='exact').pvalue),
                       mixed_log_ratio=est, p_mixed=p))
CK = pd.DataFrame(ck)
for col in ['p_domain_wilcoxon', 'p_mixed']: CK[col + '_holm'] = multipletests(CK[col], method='holm')[1]
CK.to_csv(f'{OUT}/tab_stats_checks.csv', index=False); print(CK.round(4).to_string(index=False))

In [ ]:
%%writefile 16_checks_causal.py
"""Step 16. Deployable (causal) reference-check schedules and an alarm operating curve for the certificate.
(a) Online evaluation: at cycle t only predictions p_0..p_t and checks performed at cycles <= t are known.
    Online estimate at t = last value of the monotone projection of (p + c)[0..t], where c interpolates the
    measured residuals y_s - p_s at past checks and is held constant after the last check (zero before the first).
    Error = RMS over 40 evenly spaced evaluation cycles that are not check cycles.
    Schedules with a budget of K checks (K = 1..5), none uses the target cell's life length:
      fixed-interval : checks at cycles j * L_src / (K + 1), j = 1..K, with L_src the median life of source-domain cells;
      event-triggered: check when the upward departure of the prediction from its running minimum, averaged over the
                       last w = max(5, L_src / 40) cycles, exceeds tau = 1% SOH, at least L_src / (2(K+1)) cycles after
                       the previous check; a check is forced when L_src / (K + 1) cycles have passed without one;
      periodic-known-life (reference, not deployable): checks at j * n / (K + 1) with the true life n.
(b) Alarm operating curve: raise an alarm for a cell when LB >= theta (offline over the whole life, and online at mid-life using the first half of the trajectory);
    report alarm rate, precision and recall for the events RMSE >= theta and RMSE >= 2%."""
import numpy as np, pandas as pd
from common import project, rms, OUT
P = pd.read_csv(f'{OUT}/pred_gbr_self.csv.gz'); P['dom'] = P.cell.str.split('/').str[0]
life = P.groupby('cell').size(); ldom = P.groupby('cell').dom.first()
TAU = 0.01


def online_err(p, y, checks, ev):
    checks = np.sort(np.asarray(checks, int)); err = []
    for t in ev:
        s = checks[checks <= t]
        if len(s): c = np.interp(np.arange(t + 1), s, y[s] - p[s]); c[:s[0]] = 0.0
        else: c = np.zeros(t + 1)
        err.append(project(p[:t + 1] + c)[-1] - y[t])
    return rms(err)


def event_schedule(p, K, L):
    w = max(5, L // 40); gap_min = max(1, L // (2 * (K + 1))); gap_max = max(1, L // (K + 1))
    dep = pd.Series(p - np.minimum.accumulate(p)).rolling(w, min_periods=1).mean().values
    sel, last = [], 0
    for t in range(1, len(p)):
        if len(sel) >= K: break
        if (dep[t] > TAU and t - last >= gap_min) or t - last >= gap_max: sel.append(t); last = t
    return sel


rows = []
for c, g in P.groupby('cell', sort=False):
    p, y = g.p.values, g.soh.values; n = len(p); d = ldom[c]
    L = int(np.median(life[ldom != d]))
    ev_all = np.unique(np.linspace(10, n - 1, 40).astype(int))
    rec = dict(cell=c, dom=d, n=n, L_src=L)
    rec['none'] = online_err(p, y, [], ev_all)
    for K in range(1, 6):
        sch = {'fixed': [int(j * L / (K + 1)) for j in range(1, K + 1) if j * L / (K + 1) < n],
               'event': event_schedule(p, K, L),
               'periodic_known': list(np.unique(np.linspace(0, n - 1, K + 2)[1:-1].astype(int)))}
        for nm, s in sch.items():
            ev = np.setdiff1d(ev_all, s)
            rec[f'{nm}{K}'] = online_err(p, y, s, ev); rec[f'{nm}{K}_used'] = len(s)
    rows.append(rec)
R = pd.DataFrame(rows); R.to_csv(f'{OUT}/cells_checks_causal.csv', index=False)
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
summ = [dict(K=0, schedule='none', median=R.none.median(), domain_weighted=R.groupby('dom').none.median().mean(), checks_used=0)]
for K in range(1, 6):
    for nm in ['fixed', 'event', 'periodic_known']:
        dm = R.groupby('dom')[list(dict.fromkeys([f'{nm}{K}', f'fixed{K}', 'none']))].median()
        r = dict(K=K, schedule=nm, median=R[f'{nm}{K}'].median(), domain_weighted=dm[f'{nm}{K}'].mean(),
                 checks_used=R[f'{nm}{K}_used'].mean(), domains_better_than_none=int((dm[f'{nm}{K}'] < dm.none).sum()),
                 p_vs_none_domain=float(wilcoxon(dm[f'{nm}{K}'], dm.none, method='exact').pvalue))
        if nm == 'event':
            r['domains_better_than_fixed'] = int((dm[f'event{K}'] < dm[f'fixed{K}']).sum())
            r['p_vs_fixed_domain'] = float(wilcoxon(dm[f'event{K}'], dm[f'fixed{K}'], method='exact').pvalue)
        summ.append(r)
S = pd.DataFrame(summ)
m = S.p_vs_none_domain.notna(); S.loc[m, 'p_vs_none_holm'] = multipletests(S.loc[m, 'p_vs_none_domain'], method='holm')[1]
m = S.p_vs_fixed_domain.notna(); S.loc[m, 'p_vs_fixed_holm'] = multipletests(S.loc[m, 'p_vs_fixed_domain'], method='holm')[1]
S.to_csv(f'{OUT}/tab_checks_causal.csv', index=False); print(S.round(4).to_string(index=False))

# ---------------- (b) alarm operating curve ----------------
C = pd.read_csv(f'{OUT}/cells_gbr_self.csv'); on = []
for c, g in P.groupby('cell', sort=False):
    h = max(10, len(g) // 2); p, y = g.p.values[:h], g.soh.values[:h]; q = project(p); on.append(dict(cell=c, LB_on=rms(p - q), rmse_on=rms(p - y)))
C = C.merge(pd.DataFrame(on), on='cell'); al = []
for mode, lb, err in [('offline', 'LB', 'rmse'), ('online, first half of life', 'LB_on', 'rmse_on')]:
    for th in [0.0025, 0.005, 0.0075, 0.01, 0.015, 0.02]:
        a = C[lb] >= th
        for ev_name, ev in [('RMSE >= theta', C[err] >= th), ('RMSE >= 2%', C[err] >= 0.02)]:
            al.append(dict(mode=mode, theta=th, event=ev_name, alarm_rate=a.mean(), prevalence=ev.mean(),
                           precision=(ev[a].mean() if a.any() else np.nan), recall=(a[ev].mean() if ev.any() else np.nan)))
A = pd.DataFrame(al); A.to_csv(f'{OUT}/tab_alarm.csv', index=False); print(A.round(3).to_string(index=False))

In [ ]:
%%writefile 17_conformal.py
"""Step 17. Conformal error bounds that do not assume exchangeability of cells across domains.
Same nested leave-one-domain-out set-up and signals as step 4 (meta-model GBR on log signals -> log RMSE,
conformity score = log RMSE - prediction, scores from an inner leave-one-domain-out on the 12 source domains).
Variants (target alpha = 0.10):
  split          : pooled split conformal (step 4, repeated for comparison);
  weighted       : weighted conformal under covariate shift (Tibshirani et al., 2019); likelihood-ratio weights from a
                   logistic domain classifier that separates source cells from target cells on the label-free signals;
  family         : group-conditional calibration with scores from source domains of the same dataset family
                   (XJTU, TJU, MIT; metadata known at deployment); pooled scores when fewer than two such domains exist;
  hierarchical   : domain-level calibration: the 90% quantile of scores inside each source domain, then the
                   ceil(0.9 (m + 1))-th smallest of these m = 12 values (two-layer hierarchical conformal prediction).
Per-domain coverage with Clopper-Pearson 95% intervals; width = median ratio of the bound to the realised RMSE."""
import numpy as np, pandas as pd, warnings
from scipy.stats import beta
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from common import OUT
warnings.filterwarnings('ignore')
C = pd.read_csv(f'{OUT}/cells_gbr_self.csv')
C = C.assign(lLB=np.log(C.LB + 1e-6), lens=np.log(C.ens), lknn=np.log(C.knn), y=np.log(C.rmse))
FE = ['lLB', 'lens', 'lknn', 'rebound', 'rough']; A = 0.10
mk = lambda: HistGradientBoostingRegressor(max_iter=150, max_depth=3, learning_rate=0.05, min_samples_leaf=10)


def wquant(s, w, wt, a):
    """Weighted (1-a) quantile of scores s with weights w and test weight wt (mass at +inf)."""
    o = np.argsort(s); s, w = s[o], w[o]; cw = np.cumsum(w) / (w.sum() + wt)
    i = np.searchsorted(cw, 1 - a)
    return s[i] if i < len(s) else np.inf


out = []
for t in sorted(C.dom.unique()):
    tr, te = C[C.dom != t], C[C.dom == t]
    sc, sdom = [], []
    for u in tr.dom.unique():
        a, b = tr[tr.dom != u], tr[tr.dom == u]
        sc += list(b.y - mk().fit(a[FE], a.y).predict(b[FE])); sdom += [u] * len(b)
    sc, sdom = np.array(sc), np.array(sdom)
    pr = mk().fit(tr[FE], tr.y).predict(te[FE]); n = len(sc)
    q_split = np.quantile(sc, min(1, np.ceil((1 - A) * (n + 1)) / n))
    # weighted
    sx = StandardScaler().fit(pd.concat([tr[FE], te[FE]]))
    clf = LogisticRegression(C=1.0, max_iter=1000).fit(sx.transform(pd.concat([tr[FE], te[FE]])), np.r_[np.zeros(len(tr)), np.ones(len(te))])
    lr = lambda X: (lambda p: p / (1 - p))(np.clip(clf.predict_proba(sx.transform(X))[:, 1], 1e-4, 1 - 1e-4)) * len(tr) / len(te)
    w_src = lr(tr[FE]); w_te = lr(te[FE]); q_w = np.array([wquant(sc, w_src, wt, A) for wt in w_te])
    # family-conditional
    fam = t.split('-')[0]; same = np.array([d.split('-')[0] == fam for d in sdom])
    sf = sc[same] if len(set(sdom[same])) >= 2 else sc
    q_f = np.quantile(sf, min(1, np.ceil((1 - A) * (len(sf) + 1)) / len(sf)))
    # hierarchical (domain-level)
    dq = np.sort([np.quantile(sc[sdom == u], 1 - A) for u in np.unique(sdom)]); m = len(dq)
    k = int(np.ceil((1 - A) * (m + 1))); q_h = dq[k - 1] if k <= m else np.inf
    out.append(te.assign(pred=np.exp(pr), UB_split=np.exp(pr + q_split), UB_weighted=np.exp(pr + q_w),
                         UB_family=np.exp(pr + q_f), UB_hier=np.exp(pr + q_h), wmax=float(np.max(w_te / (w_src.sum() + w_te)))))
O = pd.concat(out); O.to_csv(f'{OUT}/cells_conformal.csv', index=False)


def cp(k, n):
    return beta.ppf(.025, k, n - k + 1) if k > 0 else 0.0, beta.ppf(.975, k + 1, n - k) if k < n else 1.0


rows, per = [], []
for v in ['split', 'weighted', 'family', 'hier']:
    cov = O.rmse <= O[f'UB_{v}']; cd = cov.groupby(O.dom).mean()
    rows.append(dict(variant=v, coverage=cov.mean(), domain_mean_coverage=cd.mean(), min_domain=cd.min(), min_domain_name=cd.idxmin(),
                     domains_below_80=int((cd < 0.8).sum()), width_median=float(np.median(np.minimum(O[f'UB_{v}'], 10) / O.rmse)),
                     infinite_bounds=float(np.isinf(O[f'UB_{v}']).mean())))
    for d, g in O.groupby('dom'):
        k = int((g.rmse <= g[f'UB_{v}']).sum()); lo, hi = cp(k, len(g))
        per.append(dict(variant=v, dom=d, cells=len(g), coverage=k / len(g), lo=lo, hi=hi))
T = pd.DataFrame(rows); T.to_csv(f'{OUT}/tab_conformal.csv', index=False); print(T.round(3).to_string(index=False))
Pd = pd.DataFrame(per); Pd.to_csv(f'{OUT}/tab_conformal_domains.csv', index=False)
print(Pd.pivot(index='dom', columns='variant', values='coverage').round(2).to_string())

In [ ]:
%%writefile 18_xai_error.py
"""Step 18. Error-centred explanation and stability null for the main configuration (GBR, self-referenced).
(a) Oracle-difference decomposition of the domain bias. For each held-out domain an oracle model with the same
    hyperparameters is cross-fitted on the target domain itself (3-fold split by cell; labels used for analysis only).
    With path-dependent TreeSHAP, f(x) = phi_0 + sum_j phi_j(x) for each model, so over target cycles
        mean(f_src - f_oracle) = (phi0_src - phi0_oracle) + sum_j [mean phi_j,src - mean phi_j,oracle].
    The first term is the difference of the models' expected outputs (it reflects the different label levels of the
    training data, i.e. label shift); the feature terms show which inputs the deployed model uses differently from a
    model that knows the target domain. Since f_oracle approximates y, the left side approximates the estimator bias.
(b) Stability null: Kendall's W of the 13 leave-one-domain-out importance rankings against (i) a permutation null that
    shuffles feature labels independently per model and (ii) models trained on disjoint halves of the 13 domains
    (20 random splits), whose importance correlation measures stability across genuinely different training data."""
import warnings, numpy as np, pandas as pd, shap
from itertools import combinations
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from common import load_cells, OUT, SEED, QUICK
warnings.filterwarnings('ignore')
D, F = load_cells(); X = ['d_' + c for c in F]; doms = sorted(D.dom.unique())
NS = 600 if QUICK else 1500; rng = np.random.default_rng(SEED)
mk = lambda: HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, random_state=SEED)
rows, terms = [], {}
for tgt in doms:
    tr = D[D.dom != tgt].iloc[::6]; te = D[D.dom == tgt].reset_index(drop=True)
    mu, sd = tr[X].mean(), tr[X].std() + 1e-12
    Ztr, Zte = (tr[X] - mu) / sd, (te[X] - mu) / sd
    m = mk().fit(Ztr, tr.soh)
    sub = np.sort(rng.choice(len(te), min(NS, len(te)), replace=False))
    ex = shap.TreeExplainer(m); s_src = ex.shap_values(Zte.iloc[sub]); b_src = float(np.ravel(ex.expected_value)[0])
    p_src = m.predict(Zte)
    # cross-fitted oracle on the target domain (split by cell)
    p_or = np.zeros(len(te)); s_or = np.zeros_like(s_src); b_or = np.zeros(len(sub))
    k = min(3, te.cell.nunique())
    for a, b in GroupKFold(k).split(Zte, groups=te.cell):
        mo = mk().fit(Zte.iloc[a][::2], te.soh.values[a][::2]); p_or[b] = mo.predict(Zte.iloc[b])
        eo = shap.TreeExplainer(mo); inb = np.isin(sub, b)
        if inb.any(): s_or[inb] = eo.shap_values(Zte.iloc[sub[inb]]); b_or[inb] = float(np.ravel(eo.expected_value)[0])
    dphi = s_src.mean(0) - s_or.mean(0); base = b_src - b_or.mean()
    bias = float(np.mean(p_src - te.soh.values)); gap = float(np.mean(p_src[sub] - p_or[sub]))
    terms[tgt] = dphi
    # old attribution (prediction shift source -> target) for comparison
    s_trn = ex.shap_values(Ztr.iloc[rng.choice(len(Ztr), NS, replace=False)]); shift = s_src.mean(0) - s_trn.mean(0)
    rows.append(dict(dom=tgt, cells=te.cell.nunique(), bias=bias, oracle_rmse=float(np.sqrt(np.mean((p_or - te.soh.values) ** 2))),
                     gap_src_minus_oracle=gap, base_term=base, feature_terms=float(dphi.sum()),
                     base_share=abs(base) / (abs(base) + np.abs(dphi).sum()),
                     top_error_feature=F[int(np.argmax(np.abs(dphi)))], top_error_share=float(np.abs(dphi).max() / np.abs(dphi).sum()),
                     top_shift_feature=F[int(np.argmax(np.abs(shift)))],
                     rho_error_vs_shift_attr=spearmanr(np.abs(dphi), np.abs(shift)).correlation))
    print(rows[-1], flush=True)
R = pd.DataFrame(rows); R.to_csv(f'{OUT}/xai_error_domains.csv', index=False)
pd.DataFrame(terms, index=F).T.to_csv(f'{OUT}/xai_error_terms.csv')

# (b) stability null
I = pd.read_csv(f'{OUT}/xai_importance.csv', index_col=0)


def kendall_w(M):
    r = np.argsort(np.argsort(-M, 1), 1) + 1.0; m_, n_ = r.shape; Rj = r.sum(0)
    return 12 * np.sum((Rj - Rj.mean()) ** 2) / (m_ ** 2 * (n_ ** 3 - n_))


W = kendall_w(I.values)
null = [kendall_w(np.array([rng.permutation(row) for row in I.values])) for _ in range(2000)]
half = []
for s in range(4 if QUICK else 20):
    r2 = np.random.default_rng(100 + s); perm = r2.permutation(doms); A_, B_ = perm[:6], perm[6:]
    imps = []
    for grp in (A_, B_):
        tr = D[D.dom.isin(grp)].iloc[::6]; mu, sd = tr[X].mean(), tr[X].std() + 1e-12; Z = (tr[X] - mu) / sd
        mm = mk().fit(Z, tr.soh); ev = Z.iloc[r2.choice(len(Z), NS, replace=False)]
        imps.append(np.abs(shap.TreeExplainer(mm).shap_values(ev)).mean(0))
    half.append(spearmanr(imps[0], imps[1]).correlation)
lodo_pairs = [spearmanr(I.iloc[a], I.iloc[b]).correlation for a, b in combinations(range(len(I)), 2)]
S = dict(kendall_W=W, W_null_mean=float(np.mean(null)), W_null_p=float((np.sum(np.array(null) >= W) + 1) / (len(null) + 1)),
         rho_lodo_pairs_mean=float(np.mean(lodo_pairs)), rho_disjoint_halves_mean=float(np.mean(half)),
         rho_disjoint_halves_min=float(np.min(half)), rho_disjoint_halves_max=float(np.max(half)),
         median_base_share=float(R.base_share.median()), median_oracle_rmse=float(R.oracle_rmse.median()),
         rho_bias_vs_gap=spearmanr(R.bias, R.gap_src_minus_oracle).correlation,
         top_feature_agreement=float((R.top_error_feature == R.top_shift_feature).mean()),
         median_rho_error_vs_shift=float(R.rho_error_vs_shift_attr.median()))
pd.DataFrame([S]).to_csv(f'{OUT}/tab_xai_error.csv', index=False); print(pd.Series(S).round(4))
print(R[['dom', 'bias', 'oracle_rmse', 'base_term', 'feature_terms', 'base_share', 'top_error_feature', 'top_shift_feature']].round(4).to_string(index=False))

In [ ]:
%%writefile 19_diag.py
"""Step 19. Additional diagnostics (Supplementary Material).
(a) Data provenance: rows per public file before/after removal of missing or infinite values.
(b) SOH reference sensitivity: main configuration (GBR, self-referenced) re-trained with SOH = capacity / nominal
    capacity (XJTU 2.0 Ah, HUST 1.1 Ah, MIT 1.1 Ah, TJU 3.5/3.5/2.5 Ah, as in the source repository) instead of the
    median of the first five cycles; certificate metrics recomputed.
(c) MLP training diagnostics: iterations used and convergence warnings for every fold of both MLP configurations.
(d) GMP surface with autocorrelated error (AR(1), phi = 0.9) next to the independent-noise surface."""
import warnings, numpy as np, pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning
from common import load_cells, cell_rows, raw_row_counts, certificate, OUT, DATA, SEED
D, F = load_cells(); XS = ['d_' + c for c in F]; doms = sorted(D.dom.unique())

# (a)
R = raw_row_counts(); R.to_csv(f'{OUT}/data_provenance.csv', index=False)
print('rows raw %d kept %d removed %d in %d files' % (R.rows_raw.sum(), R.rows_kept.sum(), R.removed.sum(), int((R.removed > 0).sum())))

# (b)
NOM = {'XJTU': 2.0, 'HUST': 1.1, 'MIT': 1.1, 'TJU-Dataset_1': 3.5, 'TJU-Dataset_2': 3.5, 'TJU-Dataset_3': 2.5}
D['soh_nom'] = D.capacity / D.dom.map(lambda d: NOM.get(d, NOM.get(d.split('-')[0])))


def fold_nom(tgt):
    warnings.filterwarnings('ignore')
    tr = D[D.dom != tgt].iloc[::6]; te = D[D.dom == tgt]
    mu, sd = tr[XS].mean(), tr[XS].std() + 1e-12
    p = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, random_state=SEED).fit((tr[XS] - mu) / sd, tr.soh_nom).predict((te[XS] - mu) / sd)
    return cell_rows(te[['cell', 'dataset', 'k']].assign(soh=te.soh_nom.values, p=p), tgt)


C = pd.DataFrame([r for rs in Parallel(n_jobs=2)(delayed(fold_nom)(t) for t in doms) for r in rs]); C.to_csv(f'{OUT}/cells_gbr_self_nominal.csv', index=False)
B = pd.read_csv(f'{OUT}/cells_gbr_self.csv')
sens = pd.DataFrame([dict(soh_reference=nm, med_rmse=d.rmse.median(), med_rmse_q=d.rmse_q.median(), improved=(d.rmse_q <= d.rmse + 1e-12).mean(),
                          lb_valid=(d.LB <= d.rmse + 1e-12).mean(), removed=np.median(d.LB ** 2 / d.rmse ** 2),
                          max_initial_soh=D.groupby('cell').soh_nom.first().max() if nm == 'nominal' else 1.0)
                     for nm, d in [('first five cycles', B), ('nominal', C)]])
sens.to_csv(f'{OUT}/tab_soh_reference.csv', index=False); print(sens.round(4).to_string(index=False))

# (c)
diag = []
for ref in ['raw', 'self']:
    X = F if ref == 'raw' else XS
    for tgt in doms:
        tr = D[D.dom != tgt].iloc[::6]; mu, sd = tr[X].mean(), tr[X].std() + 1e-12
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter('always', ConvergenceWarning)
            m = MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=300, early_stopping=True, n_iter_no_change=10, random_state=SEED).fit((tr[X] - mu) / sd, tr.soh)
        diag.append(dict(cfg=f'mlp_{ref}', dom=tgt, n_iter=m.n_iter_, converged_warning=any(issubclass(x.category, ConvergenceWarning) for x in w),
                         best_val_score=m.best_validation_score_))
G = pd.DataFrame(diag); G.to_csv(f'{OUT}/tab_mlp_diag.csv', index=False)
print(G.groupby('cfg').agg(n_iter_min=('n_iter', 'min'), n_iter_max=('n_iter', 'max'), warnings=('converged_warning', 'sum')).to_string())

# (d)
G0 = pd.read_csv(f'{DATA}/GMP.csv'); y = (G0['Capacity(Ah)'] / G0['Capacity(Ah)'].iloc[:5].median()).values
rng = np.random.default_rng(SEED); surf = []
for b in np.linspace(0, 0.05, 11):
    for s in np.linspace(0.0025, 0.03, 12):
        for rep in range(5):
            e = np.zeros(len(y)); z = rng.normal(0, s * np.sqrt(1 - 0.81), len(y)); e[0] = rng.normal(0, s)
            for t in range(1, len(y)): e[t] = 0.9 * e[t - 1] + z[t]
            cc = certificate(y + b + e, y); surf.append(dict(bias=b, noise_sd=s, rep=rep, rmse=cc['rmse'], LB=cc['LB']))
S = pd.DataFrame(surf).groupby(['bias', 'noise_sd'])[['rmse', 'LB']].mean().reset_index(); S['removed'] = S.LB ** 2 / S.rmse ** 2
S.to_csv(f'{OUT}/gmp_surface_ar1.csv', index=False); print('AR(1) surface removed share %.3f-%.3f' % (S.removed.min(), S.removed.max()))

In [ ]:
%%writefile 20_nextenergy_assets.py
"""Step 20 - Next Energy figure files (run after 07, 08 and 12).
Copies the figures, which steps 07, 08 and 12 save under the article numbering, to figures/NextEnergy:
Fig. 1 study design, Fig. 2 unseen-domain error, Fig. 3 certificate, Fig. 4 simulation with known truth,
Fig. 5 conformal bounds, Fig. 6 reference checks, Fig. 7 explainability, Fig. 8 generalizability,
Fig. S1 reliability (Supplementary Material 1). The graphical abstract is drawn by step 21."""
import os, shutil

FIG = 'figures'; NE = os.path.join(FIG, 'NextEnergy'); os.makedirs(NE, exist_ok=True)
for n in ['Figure_1', 'Figure_2', 'Figure_3', 'Figure_4', 'Figure_5', 'Figure_6', 'Figure_7', 'Figure_8', 'Figure_S1']:
    shutil.copy(os.path.join(FIG, n + '.png'), os.path.join(NE, n + '.png'))
print('Next Energy figures written to', NE)

In [ ]:
%%writefile 21_graphical_abstract.py
"""Step 21. Graphical abstract of the Next Energy article (the file submitted with the manuscript).
Drawn with matplotlib (vector shapes and text) from code; no image-generation model was used.
Uses the Liberation Sans fonts shipped in fonts/ (Arial-compatible, SIL Open Font License) and the figure style of common.py.
Output: figures/NextEnergy/Graphical_abstract.tif (3137 x 1254 px, 600 dpi, RGB; the file to upload), .pdf, .eps, .png"""
import os
import matplotlib
matplotlib.use("Agg")
import common  # noqa: F401  registers the bundled fonts and applies the figure style
from common import FIG
os.makedirs(os.path.join(FIG, "NextEnergy"), exist_ok=True)
import matplotlib.font_manager as fm
if not any(f.name == "Liberation Sans" for f in fm.fontManager.ttflist):
    raise RuntimeError("Liberation Sans not found in fonts/; the text will not fit the panels.")

# ---- 2. Draw ---------------------------------------------------------------
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle, Polygon, Circle
from sklearn.isotonic import IsotonicRegression

OUT = os.path.join(FIG, "NextEnergy", "Graphical_abstract")   # output file prefix

plt.rcParams.update({
    "font.family": "Liberation Sans",
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.linewidth": 0.6,
})

# ------------------------------------------------------------ palette
INK, MUTED, RULE = "#1f2328", "#5b6470", "#c9ced6"
ORANGE, BLUE = "#D9541E", "#2462C4"     # prediction p / projection q
GREEN, RED = "#1B8A5A", "#B42318"       # status: holds / fails (with glyph)
BG_SOFT = "#f3f4f6"                     # context panels
BG_BLUE = "#edf3fc"                     # approach (centre of gravity)
BG_FIND = "#ffffff"
BG_GREEN = "#eaf5ef"

W, H = 13.28, 5.31
fig = plt.figure(figsize=(W / 2.54, H / 2.54))
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, W); ax.set_ylim(0, H); ax.set_aspect("equal"); ax.axis("off")

TEXTS = []  # (artist, panel_x0, panel_x1) for the overflow check


def T(x, y, s, size=7, panel=None, weight="normal", color=INK, ha="center",
      va="center", style="normal", **kw):
    t = ax.text(x, y, s, fontsize=size, fontweight=weight, color=color,
                ha=ha, va=va, fontstyle=style, zorder=8, **kw)
    if panel is not None:
        TEXTS.append((t, panel))
    return t


def panel(x0, x1, label, bg, edge=None, lw=0.0):
    box = FancyBboxPatch((x0, 0.12), x1 - x0, H - 0.24,
                         boxstyle="round,pad=0,rounding_size=0.12",
                         fc=bg, ec=edge if edge else bg, lw=lw, zorder=0)
    ax.add_patch(box)
    T(x0 + 0.14, H - 0.32, label, size=7.5, weight="bold",
      color=edge if edge else MUTED, ha="left", panel=(x0, x1))
    ax.plot([x0 + 0.14, x1 - 0.14], [H - 0.52, H - 0.52], color=RULE,
            lw=0.6, zorder=1)
    return (x0, x1)


def chevron(xc, yc=2.62):
    ax.add_patch(Polygon([[xc - 0.06, yc + 0.17], [xc + 0.08, yc],
                          [xc - 0.06, yc - 0.17]], closed=True, fc=MUTED,
                         ec="none", zorder=3))


def battery(xc, yc, w, h, fill, edge=INK, level=0.0, lw=0.8):
    ax.add_patch(FancyBboxPatch((xc - w / 2, yc - h / 2), w, h,
                 boxstyle="round,pad=0,rounding_size=0.05", fc="white",
                 ec=edge, lw=lw, zorder=4))
    ax.add_patch(Rectangle((xc - w * 0.22, yc + h / 2), w * 0.44, h * 0.09,
                 fc=edge, ec="none", zorder=4))
    if level > 0:
        pad = 0.045
        ax.add_patch(Rectangle((xc - w / 2 + pad, yc - h / 2 + pad),
                     w - 2 * pad, (h - 2 * pad) * level, fc=fill,
                     ec="none", zorder=5))


def tick_glyph(xc, yc, r=0.11, color=GREEN):
    ax.add_patch(Circle((xc, yc), r, fc=color, ec="none", zorder=6))
    ax.plot([xc - 0.5 * r, xc - 0.12 * r, xc + 0.55 * r],
            [yc + 0.02 * r, yc - 0.4 * r, yc + 0.45 * r], color="white",
            lw=1.0, solid_capstyle="round", solid_joinstyle="round", zorder=7)


def cross_glyph(xc, yc, r=0.11, color=RED):
    ax.add_patch(Circle((xc, yc), r, fc=color, ec="none", zorder=6))
    d = 0.42 * r
    ax.plot([xc - d, xc + d], [yc - d, yc + d], color="white", lw=1.0,
            solid_capstyle="round", zorder=7)
    ax.plot([xc - d, xc + d], [yc + d, yc - d], color="white", lw=1.0,
            solid_capstyle="round", zorder=7)


# ------------------------------------------------------------ layout (cm)
GAP, MARG = 0.16, 0.06
widths = [1.80, 2.20, 2.75, 3.75, 2.02]
x = MARG
bounds = []
for w_ in widths:
    bounds.append((x, x + w_)); x += w_ + GAP
P = panel(*bounds[0], "Problem", "white")
I = panel(*bounds[1], "Input", "white")
M = panel(*bounds[2], "Approach", BG_BLUE, edge=BLUE, lw=0.9)
F = panel(*bounds[3], "Key findings", "#f5f6f8", edge="#3b4350", lw=0.8)
C = panel(*bounds[4], "Contribution", "white", edge=None)
C_hdr = TEXTS[-1][0]; C_hdr.set_color(GREEN)
for a, b in zip(bounds[:-1], bounds[1:]):
    chevron((a[1] + b[0]) / 2)

# ------------------------------------------------------------ 1 Problem
pc = (P[0] + P[1]) / 2
battery(pc - 0.36, 4.12, 0.38, 0.66, fill=RULE, edge=INK, level=0.0)
T(pc - 0.36, 4.09, "?", size=11, weight="bold", color=ORANGE, panel=P)
# capacity meter, struck through = capacity not measured
gx, gy, gr = pc + 0.36, 3.98, 0.27
th = np.linspace(np.pi, 0, 50)
ax.plot(gx + gr * np.cos(th), gy + gr * np.sin(th), color=INK, lw=0.8,
        zorder=5)
ax.plot([gx - gr, gx + gr], [gy, gy], color=INK, lw=0.8, zorder=5)
for a_ in np.linspace(np.pi * 0.85, np.pi * 0.15, 5):
    ax.plot([gx + 0.78 * gr * np.cos(a_), gx + gr * np.cos(a_)],
            [gy + 0.78 * gr * np.sin(a_), gy + gr * np.sin(a_)], color=INK,
            lw=0.5, zorder=5)
ax.plot([gx, gx + 0.13], [gy, gy + 0.16], color=INK, lw=0.9, zorder=5)
ax.plot([gx - 0.3, gx + 0.3], [gy - 0.12, gy + 0.36], color=RED, lw=1.3,
        solid_capstyle="round", zorder=6)

T(pc, 3.22, "SOH estimator", size=7, panel=P)
T(pc, 2.96, "on new cells", size=7, panel=P)
T(pc, 2.36, "capacity not", size=7.5, weight="bold", panel=P)
T(pc, 2.08, "measured", size=7.5, weight="bold", panel=P)
T(pc, 1.42, "What can be", size=7.5, weight="bold", color=ORANGE, panel=P)
T(pc, 1.15, "known about", size=7.5, weight="bold", color=ORANGE, panel=P)
T(pc, 0.88, "the error?", size=7.5, weight="bold", color=ORANGE, panel=P)

# ------------------------------------------------------------ 2 Input
ic = (I[0] + I[1]) / 2
for k, (lab, lev) in enumerate([("NCM", 0.78), ("NCA", 0.6), ("LFP", 0.9)]):
    xc = ic + (k - 1) * 0.6
    battery(xc, 4.18, 0.3, 0.56, fill="#9aa3ae", edge=INK, level=lev)
    T(xc, 3.69, lab, size=7, weight="bold", panel=I)

T(ic, 3.22, "387 cells", size=11, weight="bold", panel=I)
T(ic, 2.84, "13 domains", size=7.5, panel=I)
T(ic, 2.57, "310,554 cycles", size=7.5, panel=I)

sq, gp = 0.112, 0.033
x_start = ic - (13 * sq + 12 * gp) / 2
held = 9
for k in range(13):
    ax.add_patch(Rectangle((x_start + k * (sq + gp), 1.93), sq, sq,
                 fc=ORANGE if k == held else "#aab2bc", ec="none", zorder=4))
T(x_start + held * (sq + gp) + sq / 2, 2.19, "test", size=7, weight="bold",
  color=ORANGE, panel=I)
T(x_start, 2.19, "train", size=7, color=MUTED, ha="left", panel=I)
T(ic, 1.7, "leave-one-out", size=7, panel=I)
T(ic, 1.45, "by domain", size=7, panel=I)

T(ic, 0.96, "11 estimators", size=7.5, weight="bold", panel=I)
T(ic, 0.67, "ridge regression", size=7, color=MUTED, panel=I)
T(ic, 0.42, "→ neural networks", size=7, color=MUTED, panel=I)

# ------------------------------------------------------------ 3 Approach
mc = (M[0] + M[1]) / 2
T(mc, 4.56, "Monotone projection", size=7.5, weight="bold", color=BLUE,
  panel=M)

px0, px1, py0, py1 = M[0] + 0.36, M[1] - 0.14, 2.26, 4.3
ax_in = fig.add_axes([px0 / W, py0 / H, (px1 - px0) / W, (py1 - py0) / H])
n = 160
t = np.linspace(0, 1, n)
y_true = 0.52 - 0.22 * t - 0.16 * t ** 2
rng = np.random.default_rng(3)
bias = 0.07 + 0.015 * t
wig = 0.008 * rng.standard_normal(n)
bumps = np.zeros(n)
for c0, amp in [(0.18, 0.085), (0.45, 0.075), (0.7, 0.09)]:
    m = (t >= c0) & (t < c0 + 0.12)
    bumps[m] += amp * np.exp(-(t[m] - c0) / 0.045)
p = y_true + bias + wig + bumps
q = IsotonicRegression(increasing=False).fit_transform(np.arange(n), p)
ax_in.fill_between(t, p, q, where=p > q, color="#f3bfa6", lw=0,
                   interpolate=True)
ax_in.plot(t, y_true, color=MUTED, lw=1.0, ls=(0, (3, 2)))
ax_in.plot(t, p, color=ORANGE, lw=0.9)
ax_in.plot(t, q, color=BLUE, lw=1.3)
ax_in.set_xlim(0, 1); ax_in.set_ylim(0.08, 1.0)
ax_in.set_xticks([]); ax_in.set_yticks([])
for s_ in ("top", "right"):
    ax_in.spines[s_].set_visible(False)
ax_in.spines["left"].set_color(MUTED); ax_in.spines["bottom"].set_color(MUTED)
ax_in.set_facecolor(BG_BLUE)
kx = 0.07
for j, (sym, lab, col, ls) in enumerate([
        ("p", "prediction", ORANGE, "-"),
        ("q", "projection", BLUE, "-"),
        ("y", "true, unknown", MUTED, (0, (3, 2)))]):
    yy = 0.93 - j * 0.115
    ax_in.plot([kx, kx + 0.09], [yy, yy], color=col, lw=1.2, ls=ls)
    t1 = ax_in.text(kx + 0.12, yy, sym, fontsize=7, fontweight="bold",
                    color=col if col != MUTED else INK, va="center",
                    ha="left", zorder=8)
    t2 = ax_in.text(kx + 0.2, yy, lab, fontsize=7, color=INK, va="center",
                    ha="left", zorder=8)
    TEXTS.extend([(t1, M), (t2, M)])
T(M[0] + 0.24, 3.0, "SOH", size=7, color=MUTED, rotation=90, panel=M)
T((px0 + px1) / 2, 2.12, "cycles", size=7, color=MUTED, panel=M)

T(mc, 1.84, "capacity does not", size=7, color=MUTED, style="italic",
  panel=M)
T(mc, 1.60, "increase over life", size=7, color=MUTED, style="italic",
  panel=M)
T(mc, 1.19, "distance p–q", size=9, weight="bold", panel=M)
T(mc, 0.88, "≤ true error", size=9, weight="bold", panel=M)
T(mc, 0.58, "provable lower bound,", size=7, panel=M)
T(mc, 0.34, "no capacity labels", size=7, panel=M)

# ------------------------------------------------------------ 4 Key findings
fx = F[0] + 0.15
rows = [4.72, 3.58, 2.44, 1.3]
for r_ in rows[1:]:
    ax.plot([F[0] + 0.12, F[1] - 0.12], [r_ + 0.05, r_ + 0.05], color=RULE,
            lw=0.6, zorder=1)

y0 = rows[0]
tick_glyph(fx + 0.11, y0 - 0.3)
T(fx + 0.31, y0 - 0.3, "4,257 / 4,257", size=10, weight="bold", ha="left",
  panel=F)
T(fx, y0 - 0.64, "cell–estimator pairs: bound held", size=7, ha="left",
  panel=F)
T(fx, y0 - 0.89, "error ↓ in 95–100% of cells", size=7, ha="left",
  color=MUTED, panel=F)

y0 = rows[1]
T(fx, y0 - 0.3, "0.4–8.7%", size=10, weight="bold", color=BLUE, ha="left",
  panel=F)
bx0, bw, bh = fx + 1.75, 1.6, 0.2
ax.add_patch(Rectangle((bx0, y0 - 0.4), bw, bh, fc="#d0d5dc", ec="none",
                       zorder=4))
ax.add_patch(Rectangle((bx0, y0 - 0.4), bw * 0.089, bh, fc=BLUE, ec="none",
                       zorder=5))
T(fx, y0 - 0.64, "of squared error removed", size=7, ha="left", panel=F)
T(fx, y0 - 0.89, "most error is hidden bias", size=7, weight="bold",
  ha="left", panel=F)

y0 = rows[2]
T(fx, y0 - 0.24, "Label-free warning signals", size=7, ha="left", panel=F)
tick_glyph(fx + 0.11, y0 - 0.55)
T(fx + 0.3, y0 - 0.55, "domains", size=7.5, weight="bold", ha="left",
  panel=F)
T(F[1] - 0.15, y0 - 0.55, "AUC up to 0.86", size=7.5, weight="bold",
  ha="right", panel=F)
cross_glyph(fx + 0.11, y0 - 0.86)
T(fx + 0.3, y0 - 0.86, "cells", size=7.5, weight="bold", ha="left", panel=F)
T(F[1] - 0.15, y0 - 0.86, "Spearman ρ ≤ 0.33", size=7.5, weight="bold",
  ha="right", panel=F)

y0 = rows[3]
T(fx, y0 - 0.3, "2.41% → 1.86% SOH", size=10, weight="bold", color=GREEN,
  ha="left", panel=F)
T(fx, y0 - 0.64, "online error with 3 scheduled", size=7, ha="left",
  panel=F)
T(fx, y0 - 0.89, "reference capacity checks", size=7, ha="left", panel=F)

# ------------------------------------------------------------ 5 Contribution
cc = (C[0] + C[1]) / 2
sx, sy = cc, 4.12
ax.add_patch(Polygon([[sx - 0.27, sy + 0.32], [sx + 0.27, sy + 0.32],
                      [sx + 0.27, sy - 0.02], [sx, sy - 0.36],
                      [sx - 0.27, sy - 0.02]], closed=True, fc=GREEN,
                     ec="none", zorder=4))
ax.plot([sx - 0.12, sx - 0.02, sx + 0.14], [sy + 0.02, sy - 0.1, sy + 0.14],
        color="white", lw=1.3, solid_capstyle="round",
        solid_joinstyle="round", zorder=5)
T(cc, 3.42, "Safe add-on", size=8, weight="bold", color=GREEN, panel=C)
T(cc, 3.12, "for any SOH", size=7, panel=C)
T(cc, 2.88, "estimator, plus", size=7, panel=C)
T(cc, 2.64, "an error alarm", size=7, panel=C)

ax.plot([C[0] + 0.3, C[1] - 0.3], [2.28, 2.28], color=RULE, lw=0.6)

battery(cc - 0.22, 1.66, 0.26, 0.46, fill=GREEN, edge=INK, level=0.6)
gx, gy, gr = cc + 0.22, 1.56, 0.21
th = np.linspace(np.pi, 0, 40)
ax.plot(gx + gr * np.cos(th), gy + gr * np.sin(th), color=INK, lw=0.8,
        zorder=5)
ax.plot([gx - gr, gx + gr], [gy, gy], color=INK, lw=0.8, zorder=5)
ax.plot([gx, gx + 0.1], [gy, gy + 0.14], color=INK, lw=0.9, zorder=6)
T(cc, 1.0, "A few capacity", size=7.5, weight="bold", panel=C)
T(cc, 0.73, "checks remain", size=7.5, weight="bold", panel=C)
T(cc, 0.46, "necessary", size=7.5, weight="bold", panel=C)

# ------------------------------------------------------------ outer border
BORDER_PT = 0.8                       # line width in points
d = BORDER_PT / 72 * 2.54 / 2         # half the line width, in cm
ax.add_patch(Rectangle((d, d), W - 2 * d, H - 2 * d, fill=False,
                       ec="#3b4350", lw=BORDER_PT, zorder=10, clip_on=False))

# ------------------------------------------------------------ checks
fig.canvas.draw()
r = fig.canvas.get_renderer()
inv = ax.transData.inverted()
bad = 0
boxes = []
for t_, (x0, x1) in TEXTS:
    bb = t_.get_window_extent(r)
    (a0, b0), (a1, b1) = inv.transform([[bb.x0, bb.y0], [bb.x1, bb.y1]])
    boxes.append((a0, b0, a1, b1, t_.get_text()))
    if a0 < x0 + 0.06 or a1 > x1 - 0.06:
        bad += 1
        print(f"OVERFLOW {t_.get_text()!r}: {a0:.2f}-{a1:.2f} in {x0}-{x1}")
    if b0 < 0.12 or b1 > H - 0.12:
        bad += 1
        print(f"OVERFLOW (vertical) {t_.get_text()!r}: {b0:.2f}-{b1:.2f}")
for i in range(len(boxes)):
    for j in range(i + 1, len(boxes)):
        A, B = boxes[i], boxes[j]
        if A[0] < B[2] and B[0] < A[2] and A[1] < B[3] and B[1] < A[3]:
            print("OVERLAP", A[4], "|", B[4])
            bad += 1
sizes = sorted({t_.get_fontsize() for t_, _ in TEXTS})
print("font sizes used:", sizes, "| problems:", bad)

fig.savefig(OUT + ".pdf")
fig.savefig(OUT + ".eps")
# raster copies: flatten onto white and save as RGB (Elsevier: RGB/CMYK)
import io
from PIL import Image
for ext, dpi in ((".png", 300), (".tif", 600)):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi)
    im = Image.open(buf)
    rgb = Image.new("RGB", im.size, "white")
    rgb.paste(im, mask=im.split()[3])
    kw = {"compression": "tiff_lzw"} if ext == ".tif" else {}
    rgb.save(OUT + ext, dpi=(dpi, dpi), **kw)
    print("saved", OUT + ext, rgb.size, rgb.mode, dpi, "dpi")

plt.close(fig)

In [ ]:
%%writefile 22_verify.py
"""Step 22. Check the regenerated tables and figures against the article.
reference/tables/*.csv hold the tables of the article exactly as printed by 13_tables.py in the reported run, and
reference/figures.json the pixel fingerprint (SHA-256 of the decoded RGBA pixels) of every figure file of the article.
Every table cell and every figure pixel is compared; differences are listed."""
import os, glob, json, hashlib
import numpy as np, pandas as pd
from PIL import Image

REF = 'reference'; ok_t = ok_f = 0; bad = []
tabs = sorted(glob.glob(os.path.join(REF, 'tables', '*.csv')))
for f in tabs:
    name = os.path.basename(f); new = os.path.join('outputs', 'tables', name)
    if not os.path.exists(new):
        bad.append(f'{name}: missing'); continue
    a = pd.read_csv(f, dtype=str, keep_default_na=False); b = pd.read_csv(new, dtype=str, keep_default_na=False)
    if a.shape != b.shape or list(a.columns) != list(b.columns):
        bad.append(f'{name}: shape or columns differ {a.shape} vs {b.shape}'); continue
    diff = [(i, c, a.at[i, c], b.at[i, c]) for c in a.columns for i in range(len(a)) if a.at[i, c] != b.at[i, c]]
    if diff:
        bad.append(f'{name}: {len(diff)} cells differ, e.g. row {diff[0][0]} "{diff[0][1]}": article {diff[0][2]} / now {diff[0][3]}')
    else:
        ok_t += 1


def fingerprint(path):
    im = np.asarray(Image.open(path).convert('RGBA'))
    return f'{im.shape[1]}x{im.shape[0]}:' + hashlib.sha256(im.tobytes()).hexdigest()


figs = json.load(open(os.path.join(REF, 'figures.json')))
for rel, fp in figs.items():
    if not os.path.exists(rel):
        bad.append(f'{rel}: missing'); continue
    now = fingerprint(rel)
    if now == fp:
        ok_f += 1
    else:
        bad.append(f'{rel}: pixels differ from the article ({now.split(":")[0]} vs {fp.split(":")[0]})')

print(f'tables identical to the article: {ok_t} of {len(tabs)}')
print(f'figures pixel-identical to the article: {ok_f} of {len(figs)}')
for b in bad:
    print('  DIFFERENT:', b)
if not bad:
    print('ALL TABLES AND FIGURES REPRODUCED EXACTLY')

In [ ]:
%%writefile run_all.py
"""Run the whole pipeline of the Next Energy article and check the result against the article.

    python run_all.py              # reference mode (default, about 10 minutes)
    python run_all.py --full       # full mode: retrain every model from the public data (about 2 hours on 2 CPU cores)

Reference mode restores the per-cycle predictions and per-cell results of the reported run (reference/outputs_part*.zip),
recomputes from them every analysis step that does not train an SOH estimator (certificate, signals, meta-model,
reference checks, controls, statistics, conformal bounds), and redraws every table and figure.
Full mode also retrains all estimators (steps 01, 09, 10, 11, 14, 18, 19) from the public data.
Both modes end with 22_verify.py, which compares every table cell and every figure pixel with the article.

Data: the public XJTU, TJU, HUST and MIT cells are downloaded in full mode (git clone of PINN4SOH, commit cc3cc50);
data/nasa_cycle_features.csv and data/GMP.csv are included. Set MONO_QUICK=1 for a fast smoke test of full mode.
Figures are written to figures/NextEnergy and tables to outputs/tables under the article numbering."""
import glob, os, sys, subprocess, time, zipfile

HERE = os.path.dirname(os.path.abspath(__file__)); os.chdir(HERE)
FULL = '--full' in sys.argv

# Same thread layout as the machine that produced the article (2 CPU cores, single-threaded OpenBLAS). Results of
# floating-point training can depend on the number of threads, so a larger machine is restricted to 2 cores.
env = dict(os.environ, OPENBLAS_NUM_THREADS='1', MPLBACKEND='Agg', PYTHONUNBUFFERED='1')
try:
    cpus = sorted(os.sched_getaffinity(0))
    if len(cpus) > 2:
        os.sched_setaffinity(0, cpus[:2]); print('using 2 of', len(cpus), 'CPU cores, as in the reported run')
    elif len(cpus) < 2:
        print('WARNING: only one CPU core; full mode may differ from the article in the last digits')
except AttributeError:
    print('NOTE: CPU affinity cannot be set on this system; use Linux or Colab for full mode')

# Library versions: the article's numbers were produced with the versions in requirements.txt.
from importlib.metadata import version, PackageNotFoundError
for line in open('requirements.txt'):
    name, want = line.strip().split('==')
    try:
        have = version(name)
    except PackageNotFoundError:
        have = 'not installed'
    if have != want:
        print(f'WARNING: {name} {have} is installed, the article used {want}; results may differ')

TRAIN = ['01_lodo', '14_strong all', '09_reliability', '10_explain', '11_generalize', '18_xai_error', '19_diag']
ANALYSIS = ['02_certificate', '03_signals', '04_meta', '05_checks', '06_controls', '15_stats', '16_checks_causal', '17_conformal']
DRAW = ['07_figures', '12_figures_trends', '13_tables', '08_workflow', '20_nextenergy_assets', '21_graphical_abstract', '22_verify']
if FULL:   # article order: estimators first, then analyses that use their predictions
    steps = ['01_lodo', '14_strong all', '02_certificate', '03_signals', '04_meta', '05_checks', '06_controls', '09_reliability',
             '10_explain', '11_generalize', '15_stats', '16_checks_causal', '17_conformal', '18_xai_error', '19_diag'] + DRAW
else:
    os.makedirs('outputs', exist_ok=True)
    parts = sorted(glob.glob(os.path.join('reference', 'outputs*.zip')))   # two parts, each small enough for GitHub
    if not parts:
        sys.exit('reference/outputs_part1.zip and outputs_part2.zip are missing (see README.md)')
    n = 0
    for part in parts:
        with zipfile.ZipFile(part) as z:
            z.extractall('.'); n += len(z.namelist())
    print('restored', n, 'files of the reported run into outputs/')
    steps = ANALYSIS + DRAW

for s in steps:
    t0 = time.time(); print('===', s, flush=True)
    a = s.split()
    r = subprocess.run([sys.executable, '-u', a[0] + '.py'] + a[1:], capture_output=True, text=True, env=env)
    print('\n'.join(l for l in r.stdout.splitlines() if 'arning' not in l)[-3000:])
    print(f'--- {s} finished in {time.time() - t0:.0f} s', flush=True)
    if r.returncode:
        print(r.stderr[-3000:]); sys.exit(r.returncode)
print('done')

In [ ]:
%%writefile requirements.txt
numpy==2.5.3
pandas==3.0.5
scipy==1.18.1
scikit-learn==1.9.1
matplotlib==3.11.2
joblib==1.6.0
shap==0.54.1
statsmodels==0.15.0
jax==0.12.1
jaxlib==0.12.1
optax==0.2.9
pillow==12.3.0

## Run
The last lines of the output come from `22_verify.py` and compare every table cell and figure pixel with the article.

In [ ]:
flag = '--full' if MODE == 'full' else ''
!python run_all.py {flag}

## Figures and tables

In [ ]:
from IPython.display import Image, HTML, display
import glob
for f in sorted(glob.glob('figures/NextEnergy/*.png')):
    print(f); display(Image(f, width=900))
display(HTML(open('outputs/tables/all_tables.html').read()))

In [ ]:
!zip -qr NE_results.zip figures/NextEnergy outputs/tables
from google.colab import files; files.download('NE_results.zip')